# Compile Season Data

Get tournament matchup matrix for specified season

In [1]:
season = 2025

playin_losers = (  # remove play-in losers from seeding data
    1384,  # St Francis PA
    1291,  # Mt St Mary's
    1361,  # San Diego St
    1400,  # Texas
)

model_path = '../data/models/mens/2025_03_16_model.pkl'
data_path = '../data/models/mens/2025_03_16_data.parquet'

season

2025

### Previous Tournament Results

In [2]:
import pandas as pd

pd.set_option('display.max_columns', 100)

df = pd.read_parquet(r'..\data\preprocessed\mens_kaggle\tournament_results.parquet')

df = df.loc[df['Season'] == season, :].reset_index(drop=True)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results
0,2025,1101,Abilene Chr,-1.0,-0.5
1,2025,1102,Air Force,-1.0,-1.0
2,2025,1103,Akron,0.0,-0.5
3,2025,1104,Alabama,4.0,2.0
4,2025,1105,Alabama A&M,-1.0,-1.0
...,...,...,...,...,...
375,2025,1476,Stonehill,-1.0,-1.0
376,2025,1477,East Texas A&M,-1.0,-1.0
377,2025,1478,Le Moyne,-1.0,-1.0
378,2025,1479,Mercyhurst,-1.0,-1.0


### Barttorvik Ratings

In [3]:
df_barttorvik = pd.read_parquet(r'..\data\preprocessed\mens_barttorvik\barttorvik.parquet')

df_barttorvik = df_barttorvik.loc[df_barttorvik['Season'] == season, :].reset_index(drop=True)

df_barttorvik

,Season,TEAM,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS
0,2025,Houston,124.5,87.6,36.9,0.983,0.882353,52.7,44.9,28.2,34.1,14.6,21.7,36.1,29.3,62.4,43.9,30.9,15.8,6.6,44.4,55.0,34.5,43.1,62.2,79.578,2.278,75.695,74.1,38.253
1,2025,Duke,127.5,90.7,36.8,0.981,0.911765,57.4,44.5,32.1,25.4,14.4,17.7,35.2,26.5,66.6,43.4,30.9,10.6,9.7,58.1,51.4,45.4,37.9,66.5,82.221,0.968,80.734,78.4,30.706
2,2025,Auburn,129.6,93.4,36.2,0.977,0.848485,55.7,46.0,33.5,39.2,13.4,17.4,34.3,30.3,68.9,47.2,29.2,16.5,7.5,55.5,40.2,40.6,34.8,68.5,81.057,2.573,34.433,74.0,45.533
3,2025,Florida,126.9,94.0,32.9,0.969,0.882353,55.0,45.3,32.6,33.0,15.0,17.0,38.1,28.8,70.8,45.9,29.6,12.3,8.6,52.4,46.4,43.6,37.3,70.3,82.097,1.887,28.619,71.8,37.736
4,2025,Alabama,127.0,96.2,30.8,0.961,0.757576,56.3,47.9,40.1,33.9,16.7,13.5,34.7,29.2,76.3,48.8,30.8,10.1,12.2,54.1,43.8,46.2,35.1,75.4,82.581,1.819,58.572,71.6,46.496
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
359,2025,Chicago St.,91.8,114.6,-22.8,0.072,0.125000,44.4,54.9,26.8,36.9,18.5,18.2,22.6,33.6,69.0,55.7,35.8,10.2,8.9,48.5,57.5,40.9,39.4,69.3,78.462,1.665,6.202,70.5,15.393
360,2025,The Citadel,93.4,117.2,-23.8,0.069,0.038462,46.9,54.5,32.8,32.8,19.5,15.2,27.2,32.0,65.5,56.3,34.7,5.7,11.6,52.2,54.5,45.2,42.7,65.3,78.922,1.756,0.200,62.3,9.793
361,2025,Arkansas Pine Bluff,95.9,122.4,-26.5,0.057,0.137931,50.3,56.3,32.8,40.5,20.4,16.8,26.3,34.6,72.5,58.5,35.7,6.3,9.2,52.2,59.5,37.9,44.9,72.3,80.380,1.633,0.200,59.0,14.013
362,2025,Coppin St.,87.8,112.6,-24.8,0.054,0.172414,44.0,55.6,36.4,37.1,21.6,20.5,26.5,33.9,68.9,56.3,36.3,5.4,13.2,50.4,57.8,32.1,38.8,68.7,79.614,1.919,0.200,67.8,13.111


In [4]:
df_spellings = pd.read_csv(
    r'..\data\unprocessed\kaggle\MTeamSpellings.csv', 
    encoding='cp1252'  # fixes issue with fancy quotes
)

df_spellings.loc[df_spellings.shape[0]] = ['fdu', 1192]

df_spellings

,TeamNameSpelling,TeamID
0,a&m-corpus chris,1394
1,a&m-corpus christi,1394
2,abilene chr,1101
3,abilene christian,1101
4,abilene-christian,1101
...,...,...
1173,youngstown st.,1464
1174,youngstown state,1464
1175,youngstown-st,1464
1176,youngstown-state,1464


In [5]:
spelling_to_id = dict(zip(df_spellings['TeamNameSpelling'], df_spellings['TeamID']))

len(spelling_to_id)

1178

In [6]:
from fuzzywuzzy.fuzz import token_sort_ratio
from fuzzywuzzy import process
from tqdm.autonotebook import tqdm

def match_names(team_spellings, new_data_teams):
    df_match = pd.DataFrame(
        [
            [
                new_data_team,
                *process.extract(
                    new_data_team,
                    team_spellings,
                    scorer=token_sort_ratio,
                    limit=1
                )[0][:2]
            ] for new_data_team in tqdm(new_data_teams)
        ],
        columns=['New Data Team', 'Team Spelling', 'Match Score']
    ).sort_values('Match Score', ignore_index=True)

    team_to_spelling = dict(zip(df_match['New Data Team'], df_match['Team Spelling']))

    return df_match, team_to_spelling

C:\Users\mhugh\AppData\Local\Temp\ipykernel_22560\2578028529.py:3: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [7]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_barttorvik['TEAM'].unique())

df_match.head(25)

  0%|          | 0/364 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,Queens,Queens (NC),80
1,UT Rio Grande Valley,texas rio grande valley,88
2,Saint Francis,saint francis (ny),90
3,Texas A&M Commerce,tx a&m commerce,91
4,Cal St. Bakersfield,cal state bakersfield,92
5,Southeast Missouri St.,southeast missouri state,93
6,Mississippi Valley St.,mississippi valley state,93
7,Texas A&M Corpus Chris,texas a&m-corpus christi,96
8,Merrimack,merrimack,100
9,Harvard,harvard,100


In [8]:
df_barttorvik.insert(1, 'TeamID', df_barttorvik['TEAM'].map(team_to_spelling).map(spelling_to_id))

df_barttorvik

,Season,TeamID,TEAM,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS
0,2025,1222,Houston,124.5,87.6,36.9,0.983,0.882353,52.7,44.9,28.2,34.1,14.6,21.7,36.1,29.3,62.4,43.9,30.9,15.8,6.6,44.4,55.0,34.5,43.1,62.2,79.578,2.278,75.695,74.1,38.253
1,2025,1181,Duke,127.5,90.7,36.8,0.981,0.911765,57.4,44.5,32.1,25.4,14.4,17.7,35.2,26.5,66.6,43.4,30.9,10.6,9.7,58.1,51.4,45.4,37.9,66.5,82.221,0.968,80.734,78.4,30.706
2,2025,1120,Auburn,129.6,93.4,36.2,0.977,0.848485,55.7,46.0,33.5,39.2,13.4,17.4,34.3,30.3,68.9,47.2,29.2,16.5,7.5,55.5,40.2,40.6,34.8,68.5,81.057,2.573,34.433,74.0,45.533
3,2025,1196,Florida,126.9,94.0,32.9,0.969,0.882353,55.0,45.3,32.6,33.0,15.0,17.0,38.1,28.8,70.8,45.9,29.6,12.3,8.6,52.4,46.4,43.6,37.3,70.3,82.097,1.887,28.619,71.8,37.736
4,2025,1104,Alabama,127.0,96.2,30.8,0.961,0.757576,56.3,47.9,40.1,33.9,16.7,13.5,34.7,29.2,76.3,48.8,30.8,10.1,12.2,54.1,43.8,46.2,35.1,75.4,82.581,1.819,58.572,71.6,46.496
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
359,2025,1152,Chicago St.,91.8,114.6,-22.8,0.072,0.125000,44.4,54.9,26.8,36.9,18.5,18.2,22.6,33.6,69.0,55.7,35.8,10.2,8.9,48.5,57.5,40.9,39.4,69.3,78.462,1.665,6.202,70.5,15.393
360,2025,1154,The Citadel,93.4,117.2,-23.8,0.069,0.038462,46.9,54.5,32.8,32.8,19.5,15.2,27.2,32.0,65.5,56.3,34.7,5.7,11.6,52.2,54.5,45.2,42.7,65.3,78.922,1.756,0.200,62.3,9.793
361,2025,1115,Arkansas Pine Bluff,95.9,122.4,-26.5,0.057,0.137931,50.3,56.3,32.8,40.5,20.4,16.8,26.3,34.6,72.5,58.5,35.7,6.3,9.2,52.2,59.5,37.9,44.9,72.3,80.380,1.633,0.200,59.0,14.013
362,2025,1164,Coppin St.,87.8,112.6,-24.8,0.054,0.172414,44.0,55.6,36.4,37.1,21.6,20.5,26.5,33.9,68.9,56.3,36.3,5.4,13.2,50.4,57.8,32.1,38.8,68.7,79.614,1.919,0.200,67.8,13.111


In [9]:
df = pd.merge(
    df,
    df_barttorvik.drop(columns=['TEAM']),
    how='left',
    on=['Season', 'TeamID']
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS
0,2025,1101,Abilene Chr,-1.0,-0.5,98.2,101.7,-3.5,0.401,0.448276,46.7,51.4,37.2,51.3,20.7,24.0,30.0,31.5,70.1,52.6,32.6,8.4,11.2,51.6,51.6,25.3,32.5,69.9,79.635,1.815,16.713,71.9,14.812
1,2025,1102,Air Force,-1.0,-1.0,98.8,109.6,-10.8,0.233,0.125000,50.1,54.6,35.4,35.4,20.8,15.7,22.7,29.0,64.7,54.8,36.3,8.1,12.4,62.9,48.6,48.5,36.2,65.1,80.552,1.633,0.200,63.5,20.594
2,2025,1103,Akron,0.0,-0.5,113.3,106.3,7.0,0.676,0.812500,55.4,49.5,26.8,33.7,16.6,17.2,33.3,29.0,72.4,50.6,31.9,9.8,7.0,58.5,44.7,45.4,38.0,71.9,77.508,2.383,16.228,74.9,10.069
3,2025,1104,Alabama,4.0,2.0,127.0,96.2,30.8,0.961,0.757576,56.3,47.9,40.1,33.9,16.7,13.5,34.7,29.2,76.3,48.8,30.8,10.1,12.2,54.1,43.8,46.2,35.1,75.4,82.581,1.819,58.572,71.6,46.496
4,2025,1105,Alabama A&M,-1.0,-1.0,92.8,114.7,-21.9,0.080,0.241379,45.1,54.3,38.6,46.7,21.9,21.0,33.9,34.6,71.8,51.8,39.0,10.7,8.7,53.4,57.8,41.6,37.5,71.5,80.365,1.912,0.200,66.7,8.320
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
375,2025,1476,Stonehill,-1.0,-1.0,100.2,114.2,-14.0,0.182,0.433333,51.1,50.9,31.4,32.7,18.3,16.2,26.4,29.4,65.7,50.1,34.9,6.9,12.6,60.4,40.9,44.5,33.3,65.9,79.514,1.609,0.200,72.1,8.108
376,2025,1477,East Texas A&M,-1.0,-1.0,95.9,111.4,-15.5,0.152,0.161290,49.2,53.8,28.0,36.5,22.5,19.9,28.1,35.9,67.8,52.6,37.4,10.7,13.4,61.6,59.4,48.3,34.6,68.2,79.365,1.997,8.201,64.4,14.821
377,2025,1478,Le Moyne,-1.0,-1.0,102.0,120.8,-18.8,0.125,0.233333,51.5,54.7,37.7,36.9,19.6,16.3,25.7,32.9,68.4,54.1,37.0,5.9,9.0,54.3,57.2,41.3,41.4,69.3,79.636,1.911,8.778,71.1,9.500
378,2025,1479,Mercyhurst,-1.0,-1.0,99.4,115.7,-16.3,0.149,0.428571,48.6,55.0,30.9,38.3,16.4,20.3,22.2,33.3,64.5,56.0,35.7,6.0,8.2,57.8,55.5,36.4,39.8,65.3,78.459,2.198,0.200,80.7,9.270


### Barttorvik Previous Seasons

In [10]:
df_barttorvik_prev = pd.read_parquet(r'..\data\preprocessed\mens_barttorvik_full_season\barttorvik_full_season.parquet')

# no longer a team, causes bugs because of name similarities to Saint Francis PA
df_barttorvik_prev = df_barttorvik_prev.loc[df_barttorvik_prev['TEAM'] != 'St. Francis NY', :].reset_index(drop=True)

df_barttorvik_prev = df_barttorvik_prev.loc[df_barttorvik_prev['Season'] == season, :].reset_index(drop=True)

df_barttorvik_prev

,Season,TEAM,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM
0,2025,Abilene Christian,0.400,-3.619,0.543000,1.526750
1,2025,Air Force,0.351,-5.822,0.351000,-5.952250
2,2025,Akron,0.588,3.203,0.612500,4.172250
3,2025,Alabama,0.915,23.580,0.914750,22.451750
4,2025,Alabama A&M,0.160,-14.507,0.152500,-14.840000
...,...,...,...,...,...,...
361,2025,Wright St.,0.553,2.124,0.551750,2.084500
362,2025,Wyoming,0.530,1.125,0.590750,3.550500
363,2025,Xavier,0.800,12.669,0.825750,14.518500
364,2025,Yale,0.723,8.782,0.673667,6.763333


In [11]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_barttorvik_prev['TEAM'].unique())

df_match.head(25)

  0%|          | 0/366 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,Queens,Queens (NC),80
1,UT Rio Grande Valley,texas rio grande valley,88
2,Saint Francis,saint francis (ny),90
3,Texas A&M Commerce,tx a&m commerce,91
4,Winston Salem St.,winston-salem-state,91
5,Cal St. Bakersfield,cal state bakersfield,92
6,Mississippi Valley St.,mississippi valley state,93
7,Southeast Missouri St.,southeast missouri state,93
8,Texas A&M Corpus Chris,texas a&m-corpus christi,96
9,Rice,rice,100


In [12]:
df_barttorvik_prev.insert(1, 'TeamID', df_barttorvik_prev['TEAM'].map(team_to_spelling).map(spelling_to_id))

df_barttorvik_prev

,Season,TeamID,TEAM,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM
0,2025,1101,Abilene Christian,0.400,-3.619,0.543000,1.526750
1,2025,1102,Air Force,0.351,-5.822,0.351000,-5.952250
2,2025,1103,Akron,0.588,3.203,0.612500,4.172250
3,2025,1104,Alabama,0.915,23.580,0.914750,22.451750
4,2025,1105,Alabama A&M,0.160,-14.507,0.152500,-14.840000
...,...,...,...,...,...,...,...
361,2025,1460,Wright St.,0.553,2.124,0.551750,2.084500
362,2025,1461,Wyoming,0.530,1.125,0.590750,3.550500
363,2025,1462,Xavier,0.800,12.669,0.825750,14.518500
364,2025,1463,Yale,0.723,8.782,0.673667,6.763333


In [13]:
df = pd.merge(
    df,
    df_barttorvik_prev.drop(columns=['TEAM']),
    how='left',
    on=['Season', 'TeamID']
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM
0,2025,1101,Abilene Chr,-1.0,-0.5,98.2,101.7,-3.5,0.401,0.448276,46.7,51.4,37.2,51.3,20.7,24.0,30.0,31.5,70.1,52.6,32.6,8.4,11.2,51.6,51.6,25.3,32.5,69.9,79.635,1.815,16.713,71.9,14.812,0.400,-3.619,0.54300,1.52675
1,2025,1102,Air Force,-1.0,-1.0,98.8,109.6,-10.8,0.233,0.125000,50.1,54.6,35.4,35.4,20.8,15.7,22.7,29.0,64.7,54.8,36.3,8.1,12.4,62.9,48.6,48.5,36.2,65.1,80.552,1.633,0.200,63.5,20.594,0.351,-5.822,0.35100,-5.95225
2,2025,1103,Akron,0.0,-0.5,113.3,106.3,7.0,0.676,0.812500,55.4,49.5,26.8,33.7,16.6,17.2,33.3,29.0,72.4,50.6,31.9,9.8,7.0,58.5,44.7,45.4,38.0,71.9,77.508,2.383,16.228,74.9,10.069,0.588,3.203,0.61250,4.17225
3,2025,1104,Alabama,4.0,2.0,127.0,96.2,30.8,0.961,0.757576,56.3,47.9,40.1,33.9,16.7,13.5,34.7,29.2,76.3,48.8,30.8,10.1,12.2,54.1,43.8,46.2,35.1,75.4,82.581,1.819,58.572,71.6,46.496,0.915,23.580,0.91475,22.45175
4,2025,1105,Alabama A&M,-1.0,-1.0,92.8,114.7,-21.9,0.080,0.241379,45.1,54.3,38.6,46.7,21.9,21.0,33.9,34.6,71.8,51.8,39.0,10.7,8.7,53.4,57.8,41.6,37.5,71.5,80.365,1.912,0.200,66.7,8.320,0.160,-14.507,0.15250,-14.84000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
375,2025,1476,Stonehill,-1.0,-1.0,100.2,114.2,-14.0,0.182,0.433333,51.1,50.9,31.4,32.7,18.3,16.2,26.4,29.4,65.7,50.1,34.9,6.9,12.6,60.4,40.9,44.5,33.3,65.9,79.514,1.609,0.200,72.1,8.108,0.066,-23.605,0.12350,-18.42700
376,2025,1477,East Texas A&M,-1.0,-1.0,95.9,111.4,-15.5,0.152,0.161290,49.2,53.8,28.0,36.5,22.5,19.9,28.1,35.9,67.8,52.6,37.4,10.7,13.4,61.6,59.4,48.3,34.6,68.2,79.365,1.997,8.201,64.4,14.821,0.123,-17.613,0.16000,-15.30800
377,2025,1478,Le Moyne,-1.0,-1.0,102.0,120.8,-18.8,0.125,0.233333,51.5,54.7,37.7,36.9,19.6,16.3,25.7,32.9,68.4,54.1,37.0,5.9,9.0,54.3,57.2,41.3,41.4,69.3,79.636,1.911,8.778,71.1,9.500,0.219,-11.555,NaN,NaN
378,2025,1479,Mercyhurst,-1.0,-1.0,99.4,115.7,-16.3,0.149,0.428571,48.6,55.0,30.9,38.3,16.4,20.3,22.2,33.3,64.5,56.0,35.7,6.0,8.2,57.8,55.5,36.4,39.8,65.3,78.459,2.198,0.200,80.7,9.270,NaN,NaN,NaN,NaN


In [14]:
df.loc[df['Past Year BARTHAG'].isna(), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM
8,2025,1109,Alliant Intl,-1.0,-1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,2025,1118,Armstrong St,-1.0,-1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,2025,1121,Augusta,-1.0,-1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,2025,1128,Birmingham So,-1.0,-1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
33,2025,1134,Brooklyn,-1.0,-1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
46,2025,1147,Centenary,-1.0,-1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
114,2025,1215,Hardin-Simmons,-1.0,-1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
115,2025,1216,Hartford,-1.0,-0.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.246,-13.283333
188,2025,1289,Morris Brown,-1.0,-1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
201,2025,1302,NE Illinois,-1.0,-1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### My Rankings

In [15]:
df_rankings = pd.read_parquet(fr'..\data\preprocessed\mens_my_rankings\my_rankings_{season}.parquet')

df_rankings.insert(0, 'Season', season)

df_rankings.drop(columns=['Strength'], inplace=True)

df_rankings

,Season,Team,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo
0,2025,Auburn,3.242428,0.343134,1.264907,0.921773,68.703152
1,2025,Florida,3.201699,0.334235,1.249459,0.915224,70.454920
2,2025,Duke,2.985720,0.381156,1.269388,0.888232,66.866622
3,2025,Tennessee,2.830807,0.293356,1.179503,0.886146,65.177363
4,2025,Houston,2.783187,0.353887,1.214051,0.860165,63.097647
...,...,...,...,...,...,...,...
359,2025,Maryland-Eastern Shore,-2.207423,-0.228611,0.941077,1.169688,66.829804
360,2025,Prairie View,-2.543046,-0.218245,0.962367,1.180612,70.477266
361,2025,Arkansas-Pine Bluff,-2.549763,-0.266402,0.942809,1.209211,72.146373
362,2025,The Citadel,-2.660858,-0.232744,0.937706,1.170450,65.501926


In [16]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_rankings['Team'].unique())

df_match.head(25)

  0%|          | 0/364 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,Auburn,auburn,100
1,South Dakota,south dakota,100
2,Massachusetts-Lowell,massachusetts lowell,100
3,Monmouth,monmouth,100
4,Colgate,colgate,100
5,Stephen F. Austin,stephen f. austin,100
6,Incarnate Word,incarnate word,100
7,Portland State,portland state,100
8,Montana State,montana state,100
9,Central Michigan,central michigan,100


In [17]:
df_rankings.insert(1, 'TeamID', df_rankings['Team'].map(team_to_spelling).map(spelling_to_id))

df_rankings

,Season,TeamID,Team,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo
0,2025,1120,Auburn,3.242428,0.343134,1.264907,0.921773,68.703152
1,2025,1196,Florida,3.201699,0.334235,1.249459,0.915224,70.454920
2,2025,1181,Duke,2.985720,0.381156,1.269388,0.888232,66.866622
3,2025,1397,Tennessee,2.830807,0.293356,1.179503,0.886146,65.177363
4,2025,1222,Houston,2.783187,0.353887,1.214051,0.860165,63.097647
...,...,...,...,...,...,...,...,...
359,2025,1271,Maryland-Eastern Shore,-2.207423,-0.228611,0.941077,1.169688,66.829804
360,2025,1341,Prairie View,-2.543046,-0.218245,0.962367,1.180612,70.477266
361,2025,1115,Arkansas-Pine Bluff,-2.549763,-0.266402,0.942809,1.209211,72.146373
362,2025,1154,The Citadel,-2.660858,-0.232744,0.937706,1.170450,65.501926


In [18]:
df_rankings.loc[df_rankings['TeamID'].isna(), :]

,Season,TeamID,Team,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo


In [19]:
df = pd.merge(
    df,
    df_rankings.drop(columns=['Team']),
    how='left',
    on=['Season', 'TeamID']
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo
0,2025,1101,Abilene Chr,-1.0,-0.5,98.2,101.7,-3.5,0.401,0.448276,46.7,51.4,37.2,51.3,20.7,24.0,30.0,31.5,70.1,52.6,32.6,8.4,11.2,51.6,51.6,25.3,32.5,69.9,79.635,1.815,16.713,71.9,14.812,0.400,-3.619,0.54300,1.52675,-0.290457,-0.057721,0.973527,1.031248,69.827777
1,2025,1102,Air Force,-1.0,-1.0,98.8,109.6,-10.8,0.233,0.125000,50.1,54.6,35.4,35.4,20.8,15.7,22.7,29.0,64.7,54.8,36.3,8.1,12.4,62.9,48.6,48.5,36.2,65.1,80.552,1.633,0.200,63.5,20.594,0.351,-5.822,0.35100,-5.95225,-1.236117,-0.141134,0.973920,1.115054,65.257013
2,2025,1103,Akron,0.0,-0.5,113.3,106.3,7.0,0.676,0.812500,55.4,49.5,26.8,33.7,16.6,17.2,33.3,29.0,72.4,50.6,31.9,9.8,7.0,58.5,44.7,45.4,38.0,71.9,77.508,2.383,16.228,74.9,10.069,0.588,3.203,0.61250,4.17225,1.091813,0.069518,1.123417,1.053899,72.412712
3,2025,1104,Alabama,4.0,2.0,127.0,96.2,30.8,0.961,0.757576,56.3,47.9,40.1,33.9,16.7,13.5,34.7,29.2,76.3,48.8,30.8,10.1,12.2,54.1,43.8,46.2,35.1,75.4,82.581,1.819,58.572,71.6,46.496,0.915,23.580,0.91475,22.45175,2.758236,0.291693,1.244913,0.953220,75.641968
4,2025,1105,Alabama A&M,-1.0,-1.0,92.8,114.7,-21.9,0.080,0.241379,45.1,54.3,38.6,46.7,21.9,21.0,33.9,34.6,71.8,51.8,39.0,10.7,8.7,53.4,57.8,41.6,37.5,71.5,80.365,1.912,0.200,66.7,8.320,0.160,-14.507,0.15250,-14.84000,-1.888423,-0.238752,0.913036,1.151789,71.202518
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
375,2025,1476,Stonehill,-1.0,-1.0,100.2,114.2,-14.0,0.182,0.433333,51.1,50.9,31.4,32.7,18.3,16.2,26.4,29.4,65.7,50.1,34.9,6.9,12.6,60.4,40.9,44.5,33.3,65.9,79.514,1.609,0.200,72.1,8.108,0.066,-23.605,0.12350,-18.42700,-0.911527,-0.134862,1.003661,1.138523,65.952891
376,2025,1477,East Texas A&M,-1.0,-1.0,95.9,111.4,-15.5,0.152,0.161290,49.2,53.8,28.0,36.5,22.5,19.9,28.1,35.9,67.8,52.6,37.4,10.7,13.4,61.6,59.4,48.3,34.6,68.2,79.365,1.997,8.201,64.4,14.821,0.123,-17.613,0.16000,-15.30800,-1.291975,-0.151314,0.950204,1.101518,68.366911
377,2025,1478,Le Moyne,-1.0,-1.0,102.0,120.8,-18.8,0.125,0.233333,51.5,54.7,37.7,36.9,19.6,16.3,25.7,32.9,68.4,54.1,37.0,5.9,9.0,54.3,57.2,41.3,41.4,69.3,79.636,1.911,8.778,71.1,9.500,0.219,-11.555,NaN,NaN,-1.463802,-0.194143,1.016376,1.210519,68.777229
378,2025,1479,Mercyhurst,-1.0,-1.0,99.4,115.7,-16.3,0.149,0.428571,48.6,55.0,30.9,38.3,16.4,20.3,22.2,33.3,64.5,56.0,35.7,6.0,8.2,57.8,55.5,36.4,39.8,65.3,78.459,2.198,0.200,80.7,9.270,NaN,NaN,NaN,NaN,-1.391770,-0.163007,0.994381,1.157388,65.296330


In [20]:
df.loc[df['Rating'].isna(), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo
8,2025,1109,Alliant Intl,-1.0,-1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,2025,1118,Armstrong St,-1.0,-1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,2025,1121,Augusta,-1.0,-1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,2025,1128,Birmingham So,-1.0,-1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
33,2025,1134,Brooklyn,-1.0,-1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
46,2025,1147,Centenary,-1.0,-1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
114,2025,1215,Hardin-Simmons,-1.0,-1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
115,2025,1216,Hartford,-1.0,-0.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.2460,-13.283333,NaN,NaN,NaN,NaN,NaN
188,2025,1289,Morris Brown,-1.0,-1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
201,2025,1302,NE Illinois,-1.0,-1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Starters

In [21]:
df_starters = pd.read_parquet(fr'..\data\preprocessed\mens_starters\starters_{season}.parquet')

df_starters.insert(0, 'Season', season)

df_starters.rename(columns={'Rating': 'Starters'}, inplace=True)

df_starters

,Season,Team,Starters
0,2025,Florida,0.625456
1,2025,Auburn,0.605368
2,2025,Duke,0.581122
3,2025,Houston,0.574558
4,2025,St. John's (NY),0.574527
...,...,...,...
359,2025,South Carolina Upstate,-0.379811
360,2025,The Citadel,-0.425557
361,2025,Prairie View,-0.436344
362,2025,Sacramento State,-0.439324


In [22]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_starters['Team'].unique())

df_match.head(25)

  0%|          | 0/364 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,Florida,florida,100
1,Evansville,evansville,100
2,Louisiana,louisiana,100
3,Hofstra,hofstra,100
4,Old Dominion,old dominion,100
5,Ohio,ohio,100
6,Virginia Military Institute,virginia military institute,100
7,Campbell,campbell,100
8,Stephen F. Austin,stephen f. austin,100
9,Rider,rider,100


In [23]:
df_starters.insert(1, 'TeamID', df_starters['Team'].map(team_to_spelling).map(spelling_to_id))

df_starters

,Season,TeamID,Team,Starters
0,2025,1196,Florida,0.625456
1,2025,1120,Auburn,0.605368
2,2025,1181,Duke,0.581122
3,2025,1222,Houston,0.574558
4,2025,1385,St. John's (NY),0.574527
...,...,...,...,...
359,2025,1367,South Carolina Upstate,-0.379811
360,2025,1154,The Citadel,-0.425557
361,2025,1341,Prairie View,-0.436344
362,2025,1170,Sacramento State,-0.439324


In [24]:
df_starters.loc[df_starters['TeamID'].isna(), :]

,Season,TeamID,Team,Starters


In [25]:
df = pd.merge(
    df,
    df_starters.drop(columns=['Team']),
    how='left',
    on=['Season', 'TeamID']
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters
0,2025,1101,Abilene Chr,-1.0,-0.5,98.2,101.7,-3.5,0.401,0.448276,46.7,51.4,37.2,51.3,20.7,24.0,30.0,31.5,70.1,52.6,32.6,8.4,11.2,51.6,51.6,25.3,32.5,69.9,79.635,1.815,16.713,71.9,14.812,0.400,-3.619,0.54300,1.52675,-0.290457,-0.057721,0.973527,1.031248,69.827777,-0.009629
1,2025,1102,Air Force,-1.0,-1.0,98.8,109.6,-10.8,0.233,0.125000,50.1,54.6,35.4,35.4,20.8,15.7,22.7,29.0,64.7,54.8,36.3,8.1,12.4,62.9,48.6,48.5,36.2,65.1,80.552,1.633,0.200,63.5,20.594,0.351,-5.822,0.35100,-5.95225,-1.236117,-0.141134,0.973920,1.115054,65.257013,-0.313992
2,2025,1103,Akron,0.0,-0.5,113.3,106.3,7.0,0.676,0.812500,55.4,49.5,26.8,33.7,16.6,17.2,33.3,29.0,72.4,50.6,31.9,9.8,7.0,58.5,44.7,45.4,38.0,71.9,77.508,2.383,16.228,74.9,10.069,0.588,3.203,0.61250,4.17225,1.091813,0.069518,1.123417,1.053899,72.412712,0.342931
3,2025,1104,Alabama,4.0,2.0,127.0,96.2,30.8,0.961,0.757576,56.3,47.9,40.1,33.9,16.7,13.5,34.7,29.2,76.3,48.8,30.8,10.1,12.2,54.1,43.8,46.2,35.1,75.4,82.581,1.819,58.572,71.6,46.496,0.915,23.580,0.91475,22.45175,2.758236,0.291693,1.244913,0.953220,75.641968,0.456281
4,2025,1105,Alabama A&M,-1.0,-1.0,92.8,114.7,-21.9,0.080,0.241379,45.1,54.3,38.6,46.7,21.9,21.0,33.9,34.6,71.8,51.8,39.0,10.7,8.7,53.4,57.8,41.6,37.5,71.5,80.365,1.912,0.200,66.7,8.320,0.160,-14.507,0.15250,-14.84000,-1.888423,-0.238752,0.913036,1.151789,71.202518,-0.313686
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
375,2025,1476,Stonehill,-1.0,-1.0,100.2,114.2,-14.0,0.182,0.433333,51.1,50.9,31.4,32.7,18.3,16.2,26.4,29.4,65.7,50.1,34.9,6.9,12.6,60.4,40.9,44.5,33.3,65.9,79.514,1.609,0.200,72.1,8.108,0.066,-23.605,0.12350,-18.42700,-0.911527,-0.134862,1.003661,1.138523,65.952891,-0.153608
376,2025,1477,East Texas A&M,-1.0,-1.0,95.9,111.4,-15.5,0.152,0.161290,49.2,53.8,28.0,36.5,22.5,19.9,28.1,35.9,67.8,52.6,37.4,10.7,13.4,61.6,59.4,48.3,34.6,68.2,79.365,1.997,8.201,64.4,14.821,0.123,-17.613,0.16000,-15.30800,-1.291975,-0.151314,0.950204,1.101518,68.366911,-0.285204
377,2025,1478,Le Moyne,-1.0,-1.0,102.0,120.8,-18.8,0.125,0.233333,51.5,54.7,37.7,36.9,19.6,16.3,25.7,32.9,68.4,54.1,37.0,5.9,9.0,54.3,57.2,41.3,41.4,69.3,79.636,1.911,8.778,71.1,9.500,0.219,-11.555,NaN,NaN,-1.463802,-0.194143,1.016376,1.210519,68.777229,-0.336253
378,2025,1479,Mercyhurst,-1.0,-1.0,99.4,115.7,-16.3,0.149,0.428571,48.6,55.0,30.9,38.3,16.4,20.3,22.2,33.3,64.5,56.0,35.7,6.0,8.2,57.8,55.5,36.4,39.8,65.3,78.459,2.198,0.200,80.7,9.270,NaN,NaN,NaN,NaN,-1.391770,-0.163007,0.994381,1.157388,65.296330,-0.166129


In [26]:
df.loc[df['Starters'].isna(), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters
8,2025,1109,Alliant Intl,-1.0,-1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,2025,1118,Armstrong St,-1.0,-1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,2025,1121,Augusta,-1.0,-1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,2025,1128,Birmingham So,-1.0,-1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
33,2025,1134,Brooklyn,-1.0,-1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
46,2025,1147,Centenary,-1.0,-1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
114,2025,1215,Hardin-Simmons,-1.0,-1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
115,2025,1216,Hartford,-1.0,-0.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.2460,-13.283333,NaN,NaN,NaN,NaN,NaN,NaN
188,2025,1289,Morris Brown,-1.0,-1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
201,2025,1302,NE Illinois,-1.0,-1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Openskill Ratings

In [27]:
df_os = pd.read_parquet(fr'..\data\preprocessed\mens_os_rankings\os_rankings_{season}.parquet')

df_os.insert(0, 'Season', season)

df_os.drop(columns=['Sigma'], inplace=True)

df_os

,Season,Team,Mu,OS Rating
0,2025,Florida,54.680187,42.528146
1,2025,Houston,53.364443,41.027495
2,2025,Duke,52.248584,39.346145
3,2025,St. John's (NY),51.065818,38.274469
4,2025,Auburn,50.620277,38.088174
...,...,...,...,...
359,2025,Maryland-Eastern Shore,3.051341,-11.355958
360,2025,Prairie View,0.107787,-12.867380
361,2025,Arkansas-Pine Bluff,0.285370,-13.009438
362,2025,The Citadel,1.512402,-13.337511


In [28]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_os['Team'].unique())

df_match.head(25)

  0%|          | 0/364 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,Florida,florida,100
1,UC Davis,uc davis,100
2,Longwood,longwood,100
3,Fordham,fordham,100
4,Cal State Bakersfield,cal state bakersfield,100
5,Marist,marist,100
6,South Florida,south florida,100
7,Texas State,texas state,100
8,Long Island University,long island university,100
9,Siena,siena,100


In [29]:
df_os.insert(1, 'TeamID', df_os['Team'].map(team_to_spelling).map(spelling_to_id))

df_os

,Season,TeamID,Team,Mu,OS Rating
0,2025,1196,Florida,54.680187,42.528146
1,2025,1222,Houston,53.364443,41.027495
2,2025,1181,Duke,52.248584,39.346145
3,2025,1385,St. John's (NY),51.065818,38.274469
4,2025,1120,Auburn,50.620277,38.088174
...,...,...,...,...,...
359,2025,1271,Maryland-Eastern Shore,3.051341,-11.355958
360,2025,1341,Prairie View,0.107787,-12.867380
361,2025,1115,Arkansas-Pine Bluff,0.285370,-13.009438
362,2025,1154,The Citadel,1.512402,-13.337511


In [30]:
df = pd.merge(
    df,
    df_os.drop(columns=['Team']),
    how='left',
    on=['Season', 'TeamID']
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters,Mu,OS Rating
0,2025,1101,Abilene Chr,-1.0,-0.5,98.2,101.7,-3.5,0.401,0.448276,46.7,51.4,37.2,51.3,20.7,24.0,30.0,31.5,70.1,52.6,32.6,8.4,11.2,51.6,51.6,25.3,32.5,69.9,79.635,1.815,16.713,71.9,14.812,0.400,-3.619,0.54300,1.52675,-0.290457,-0.057721,0.973527,1.031248,69.827777,-0.009629,20.524272,8.034226
1,2025,1102,Air Force,-1.0,-1.0,98.8,109.6,-10.8,0.233,0.125000,50.1,54.6,35.4,35.4,20.8,15.7,22.7,29.0,64.7,54.8,36.3,8.1,12.4,62.9,48.6,48.5,36.2,65.1,80.552,1.633,0.200,63.5,20.594,0.351,-5.822,0.35100,-5.95225,-1.236117,-0.141134,0.973920,1.115054,65.257013,-0.313992,8.792825,-5.530769
2,2025,1103,Akron,0.0,-0.5,113.3,106.3,7.0,0.676,0.812500,55.4,49.5,26.8,33.7,16.6,17.2,33.3,29.0,72.4,50.6,31.9,9.8,7.0,58.5,44.7,45.4,38.0,71.9,77.508,2.383,16.228,74.9,10.069,0.588,3.203,0.61250,4.17225,1.091813,0.069518,1.123417,1.053899,72.412712,0.342931,37.184009,24.282647
3,2025,1104,Alabama,4.0,2.0,127.0,96.2,30.8,0.961,0.757576,56.3,47.9,40.1,33.9,16.7,13.5,34.7,29.2,76.3,48.8,30.8,10.1,12.2,54.1,43.8,46.2,35.1,75.4,82.581,1.819,58.572,71.6,46.496,0.915,23.580,0.91475,22.45175,2.758236,0.291693,1.244913,0.953220,75.641968,0.456281,48.415068,36.555816
4,2025,1105,Alabama A&M,-1.0,-1.0,92.8,114.7,-21.9,0.080,0.241379,45.1,54.3,38.6,46.7,21.9,21.0,33.9,34.6,71.8,51.8,39.0,10.7,8.7,53.4,57.8,41.6,37.5,71.5,80.365,1.912,0.200,66.7,8.320,0.160,-14.507,0.15250,-14.84000,-1.888423,-0.238752,0.913036,1.151789,71.202518,-0.313686,4.573831,-7.827690
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
375,2025,1476,Stonehill,-1.0,-1.0,100.2,114.2,-14.0,0.182,0.433333,51.1,50.9,31.4,32.7,18.3,16.2,26.4,29.4,65.7,50.1,34.9,6.9,12.6,60.4,40.9,44.5,33.3,65.9,79.514,1.609,0.200,72.1,8.108,0.066,-23.605,0.12350,-18.42700,-0.911527,-0.134862,1.003661,1.138523,65.952891,-0.153608,13.663540,0.983780
376,2025,1477,East Texas A&M,-1.0,-1.0,95.9,111.4,-15.5,0.152,0.161290,49.2,53.8,28.0,36.5,22.5,19.9,28.1,35.9,67.8,52.6,37.4,10.7,13.4,61.6,59.4,48.3,34.6,68.2,79.365,1.997,8.201,64.4,14.821,0.123,-17.613,0.16000,-15.30800,-1.291975,-0.151314,0.950204,1.101518,68.366911,-0.285204,7.878238,-6.062353
377,2025,1478,Le Moyne,-1.0,-1.0,102.0,120.8,-18.8,0.125,0.233333,51.5,54.7,37.7,36.9,19.6,16.3,25.7,32.9,68.4,54.1,37.0,5.9,9.0,54.3,57.2,41.3,41.4,69.3,79.636,1.911,8.778,71.1,9.500,0.219,-11.555,NaN,NaN,-1.463802,-0.194143,1.016376,1.210519,68.777229,-0.336253,6.311455,-6.479544
378,2025,1479,Mercyhurst,-1.0,-1.0,99.4,115.7,-16.3,0.149,0.428571,48.6,55.0,30.9,38.3,16.4,20.3,22.2,33.3,64.5,56.0,35.7,6.0,8.2,57.8,55.5,36.4,39.8,65.3,78.459,2.198,0.200,80.7,9.270,NaN,NaN,NaN,NaN,-1.391770,-0.163007,0.994381,1.157388,65.296330,-0.166129,16.699941,3.736195


In [31]:
df.loc[df['OS Rating'].isna(), :]

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters,Mu,OS Rating
8,2025,1109,Alliant Intl,-1.0,-1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
17,2025,1118,Armstrong St,-1.0,-1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,2025,1121,Augusta,-1.0,-1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,2025,1128,Birmingham So,-1.0,-1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
33,2025,1134,Brooklyn,-1.0,-1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
46,2025,1147,Centenary,-1.0,-1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
114,2025,1215,Hardin-Simmons,-1.0,-1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
115,2025,1216,Hartford,-1.0,-0.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.2460,-13.283333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
188,2025,1289,Morris Brown,-1.0,-1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
201,2025,1302,NE Illinois,-1.0,-1.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Betting Odds

Omitted - minimally helpful and annoying to scrape for current season

In [32]:
# df_bo = pd.read_parquet('../data/preprocessed/mens_betting/betting.parquet')

# df_bo = df_bo.loc[df_bo['Season'] == season, :].reset_index(drop=True)

# df_bo

In [33]:
# df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_bo['Team'].unique())

# df_match.head(25)

In [34]:
# df_bo.insert(1, 'TeamID', df_bo['Team'].map(team_to_spelling).map(spelling_to_id))

# df_bo

In [35]:
# df = pd.merge(
#     df,
#     df_bo.drop(columns=['Team']),
#     how='left',
#     on=['Season', 'TeamID']
# )

# df

In [36]:
# df.loc[df['Implied Champion Probability'].isna(), :]

### Map to Matchups

In [37]:
# df_seeds = pd.read_csv(fr'..\data\unprocessed\kaggle\{season}_tourney_seeds.csv')

# df_seeds = df_seeds.loc[df_seeds['Tournament'] == 'M', :].reset_index(drop=True)

# df_seeds.rename(columns={'Seed': 'Region Seed'}, inplace=True)
# df_seeds.insert(2, 'Region', df_seeds['Region Seed'].str[0])
# df_seeds.insert(3, 'Seed', df_seeds['Region Seed'].str.extract('(\d+)').astype(int))

# df_seeds

In [38]:
df_seeds = pd.read_csv(r'..\data\unprocessed\kaggle\MNCAATourneySeeds.csv')

df_seeds = df_seeds.loc[df_seeds['Season'] == season, :].reset_index(drop=True)

df_seeds.insert(2, 'Play In', df_seeds['Seed'].str.endswith(('a', 'b')))
df_seeds.insert(2, 'Region', df_seeds['Seed'].str[0])
df_seeds['Seed'] = df_seeds['Seed'].str.extract('(\d+)').astype(int)

# df_seeds = df_seeds.loc[~df_seeds['TeamID'].isin(playin_losers), :].reset_index(drop=True)

df_seeds

,Season,Seed,Region,Play In,TeamID
0,2025,1,W,False,1181
1,2025,2,W,False,1104
2,2025,3,W,False,1458
3,2025,4,W,False,1112
4,2025,5,W,False,1332
...,...,...,...,...,...
63,2025,12,Z,False,1161
64,2025,13,Z,False,1213
65,2025,14,Z,False,1423
66,2025,15,Z,False,1303


In [39]:
df = df.merge(
    df_seeds,
    how='left',
    on=['Season', 'TeamID'],
)

df

,Season,TeamID,Team,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters,Mu,OS Rating,Seed,Region,Play In
0,2025,1101,Abilene Chr,-1.0,-0.5,98.2,101.7,-3.5,0.401,0.448276,46.7,51.4,37.2,51.3,20.7,24.0,30.0,31.5,70.1,52.6,32.6,8.4,11.2,51.6,51.6,25.3,32.5,69.9,79.635,1.815,16.713,71.9,14.812,0.400,-3.619,0.54300,1.52675,-0.290457,-0.057721,0.973527,1.031248,69.827777,-0.009629,20.524272,8.034226,NaN,NaN,NaN
1,2025,1102,Air Force,-1.0,-1.0,98.8,109.6,-10.8,0.233,0.125000,50.1,54.6,35.4,35.4,20.8,15.7,22.7,29.0,64.7,54.8,36.3,8.1,12.4,62.9,48.6,48.5,36.2,65.1,80.552,1.633,0.200,63.5,20.594,0.351,-5.822,0.35100,-5.95225,-1.236117,-0.141134,0.973920,1.115054,65.257013,-0.313992,8.792825,-5.530769,NaN,NaN,NaN
2,2025,1103,Akron,0.0,-0.5,113.3,106.3,7.0,0.676,0.812500,55.4,49.5,26.8,33.7,16.6,17.2,33.3,29.0,72.4,50.6,31.9,9.8,7.0,58.5,44.7,45.4,38.0,71.9,77.508,2.383,16.228,74.9,10.069,0.588,3.203,0.61250,4.17225,1.091813,0.069518,1.123417,1.053899,72.412712,0.342931,37.184009,24.282647,13.0,W,False
3,2025,1104,Alabama,4.0,2.0,127.0,96.2,30.8,0.961,0.757576,56.3,47.9,40.1,33.9,16.7,13.5,34.7,29.2,76.3,48.8,30.8,10.1,12.2,54.1,43.8,46.2,35.1,75.4,82.581,1.819,58.572,71.6,46.496,0.915,23.580,0.91475,22.45175,2.758236,0.291693,1.244913,0.953220,75.641968,0.456281,48.415068,36.555816,2.0,W,False
4,2025,1105,Alabama A&M,-1.0,-1.0,92.8,114.7,-21.9,0.080,0.241379,45.1,54.3,38.6,46.7,21.9,21.0,33.9,34.6,71.8,51.8,39.0,10.7,8.7,53.4,57.8,41.6,37.5,71.5,80.365,1.912,0.200,66.7,8.320,0.160,-14.507,0.15250,-14.84000,-1.888423,-0.238752,0.913036,1.151789,71.202518,-0.313686,4.573831,-7.827690,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
375,2025,1476,Stonehill,-1.0,-1.0,100.2,114.2,-14.0,0.182,0.433333,51.1,50.9,31.4,32.7,18.3,16.2,26.4,29.4,65.7,50.1,34.9,6.9,12.6,60.4,40.9,44.5,33.3,65.9,79.514,1.609,0.200,72.1,8.108,0.066,-23.605,0.12350,-18.42700,-0.911527,-0.134862,1.003661,1.138523,65.952891,-0.153608,13.663540,0.983780,NaN,NaN,NaN
376,2025,1477,East Texas A&M,-1.0,-1.0,95.9,111.4,-15.5,0.152,0.161290,49.2,53.8,28.0,36.5,22.5,19.9,28.1,35.9,67.8,52.6,37.4,10.7,13.4,61.6,59.4,48.3,34.6,68.2,79.365,1.997,8.201,64.4,14.821,0.123,-17.613,0.16000,-15.30800,-1.291975,-0.151314,0.950204,1.101518,68.366911,-0.285204,7.878238,-6.062353,NaN,NaN,NaN
377,2025,1478,Le Moyne,-1.0,-1.0,102.0,120.8,-18.8,0.125,0.233333,51.5,54.7,37.7,36.9,19.6,16.3,25.7,32.9,68.4,54.1,37.0,5.9,9.0,54.3,57.2,41.3,41.4,69.3,79.636,1.911,8.778,71.1,9.500,0.219,-11.555,NaN,NaN,-1.463802,-0.194143,1.016376,1.210519,68.777229,-0.336253,6.311455,-6.479544,NaN,NaN,NaN
378,2025,1479,Mercyhurst,-1.0,-1.0,99.4,115.7,-16.3,0.149,0.428571,48.6,55.0,30.9,38.3,16.4,20.3,22.2,33.3,64.5,56.0,35.7,6.0,8.2,57.8,55.5,36.4,39.8,65.3,78.459,2.198,0.200,80.7,9.270,NaN,NaN,NaN,NaN,-1.391770,-0.163007,0.994381,1.157388,65.296330,-0.166129,16.699941,3.736195,NaN,NaN,NaN


In [40]:
id_to_region = dict(zip(df['TeamID'], df['Region']))
id_to_seed = dict(zip(df['TeamID'], df['Seed']))

df_mod = pd.DataFrame(
    [
        (team_a, team_b) 
        for team_a in df['TeamID'].unique() 
        for team_b in df['TeamID'].unique() 
        if team_a != team_b
    ],
    columns=['Team A ID', 'Team B ID']
)

df_mod.insert(0, 'Season', season)
df_mod['Team A Region'] = df_mod['Team A ID'].map(id_to_region)
df_mod['Team B Region'] = df_mod['Team B ID'].map(id_to_region)
df_mod['Team A Seed'] = df_mod['Team A ID'].map(id_to_seed)
df_mod['Team B Seed'] = df_mod['Team B ID'].map(id_to_seed)

df_mod

,Season,Team A ID,Team B ID,Team A Region,Team B Region,Team A Seed,Team B Seed
0,2025,1101,1102,NaN,NaN,NaN,NaN
1,2025,1101,1103,NaN,W,NaN,13.0
2,2025,1101,1104,NaN,W,NaN,2.0
3,2025,1101,1105,NaN,NaN,NaN,NaN
4,2025,1101,1106,NaN,Y,NaN,16.0
...,...,...,...,...,...,...,...
144015,2025,1480,1475,NaN,NaN,NaN,NaN
144016,2025,1480,1476,NaN,NaN,NaN,NaN
144017,2025,1480,1477,NaN,NaN,NaN,NaN
144018,2025,1480,1478,NaN,NaN,NaN,NaN


Calculate round of matchup

In [41]:
same_region = df_mod['Team A Region'] == df_mod['Team B Region']

# round_0_condition = (df_mod['team0_playin'] == 1) & (df_mod['team1_playin'] == 1)  # no play-in games in this data

round_1_condition = df_mod['Team A Seed'] + df_mod['Team B Seed'] == 17

round_2_condition = (
    (df_mod['Team A Seed'].isin([1, 16]) & df_mod['Team B Seed'].isin([8, 9])) | 
    (df_mod['Team A Seed'].isin([8, 9]) & df_mod['Team B Seed'].isin([1, 16])) |
    (df_mod['Team A Seed'].isin([5, 12]) & df_mod['Team B Seed'].isin([4, 13])) | 
    (df_mod['Team A Seed'].isin([4, 13]) & df_mod['Team B Seed'].isin([5, 12])) |
    (df_mod['Team A Seed'].isin([6, 11]) & df_mod['Team B Seed'].isin([3, 14])) | 
    (df_mod['Team A Seed'].isin([3, 14]) & df_mod['Team B Seed'].isin([6, 11])) |
    (df_mod['Team A Seed'].isin([7, 10]) & df_mod['Team B Seed'].isin([2, 15])) | 
    (df_mod['Team A Seed'].isin([2, 15]) & df_mod['Team B Seed'].isin([7, 10]))
)

round_3_condition = (
    (df_mod['Team A Seed'].isin([1, 16, 8, 9]) & df_mod['Team B Seed'].isin([5, 12, 4, 13])) | 
    (df_mod['Team A Seed'].isin([5, 12, 4, 13]) & df_mod['Team B Seed'].isin([1, 16, 8, 9])) |
    (df_mod['Team A Seed'].isin([6, 11, 3, 14]) & df_mod['Team B Seed'].isin([7, 10, 2, 15])) | 
    (df_mod['Team A Seed'].isin([7, 10, 2, 15]) & df_mod['Team B Seed'].isin([6, 11, 3, 14]))
)

round_4_condition = (
    (df_mod['Team A Seed'].isin([1, 16, 8, 9, 5, 12, 4, 13]) & df_mod['Team B Seed'].isin([6, 11, 3, 14, 7, 10, 2, 15])) | 
    (df_mod['Team A Seed'].isin([6, 11, 3, 14, 7, 10, 2, 15]) & df_mod['Team B Seed'].isin([1, 16, 8, 9, 5, 12, 4, 13]))
)

round_5_condition = (
    (df_mod['Team A Region'].isin(['W']) & df_mod['Team B Region'].isin(['X'])) | 
    (df_mod['Team A Region'].isin(['X']) & df_mod['Team B Region'].isin(['W'])) |
    (df_mod['Team A Region'].isin(['Y']) & df_mod['Team B Region'].isin(['Z'])) | 
    (df_mod['Team A Region'].isin(['Z']) & df_mod['Team B Region'].isin(['Y']))
)

round_6_condition = (
    (df_mod['Team A Region'].isin(['W', 'X']) & df_mod['Team B Region'].isin(['Y', 'Z'])) | 
    (df_mod['Team A Region'].isin(['Y', 'Z']) & df_mod['Team B Region'].isin(['W', 'X'])) 
)

round_6_condition

0         False
1         False
2         False
3         False
4         False
          ...  
144015    False
144016    False
144017    False
144018    False
144019    False
Length: 144020, dtype: bool

In [42]:
df_mod['Round'] = float('nan')

df_mod.loc[round_6_condition, 'Round'] = 6

df_mod.loc[round_5_condition, 'Round'] = 5

df_mod.loc[round_4_condition & same_region, 'Round'] = 4

df_mod.loc[round_3_condition & same_region, 'Round'] = 3

df_mod.loc[round_2_condition & same_region, 'Round'] = 2

df_mod.loc[round_1_condition & same_region, 'Round'] = 1

df_mod

,Season,Team A ID,Team B ID,Team A Region,Team B Region,Team A Seed,Team B Seed,Round
0,2025,1101,1102,NaN,NaN,NaN,NaN,NaN
1,2025,1101,1103,NaN,W,NaN,13.0,NaN
2,2025,1101,1104,NaN,W,NaN,2.0,NaN
3,2025,1101,1105,NaN,NaN,NaN,NaN,NaN
4,2025,1101,1106,NaN,Y,NaN,16.0,NaN
...,...,...,...,...,...,...,...,...
144015,2025,1480,1475,NaN,NaN,NaN,NaN,NaN
144016,2025,1480,1476,NaN,NaN,NaN,NaN,NaN
144017,2025,1480,1477,NaN,NaN,NaN,NaN,NaN
144018,2025,1480,1478,NaN,NaN,NaN,NaN,NaN


Get Head-to-Head

In [43]:
df_h2h = pd.read_parquet('../data/preprocessed/mens_h2h/h2h.parquet')

df_h2h = df_h2h.loc[df_h2h['Season'] == season, :].reset_index(drop=True)

df_h2h

,Season,Team A,Team B,Head to Head,Common Opps
0,2025,Abilene Christian,Akron,NaN,0.109367
1,2025,Abilene Christian,Alabama,NaN,-1.428608
2,2025,Abilene Christian,Alabama A&M,NaN,1.143470
3,2025,Abilene Christian,Alabama State,NaN,0.796956
4,2025,Abilene Christian,Alcorn State,NaN,1.306308
...,...,...,...,...,...
74971,2025,Youngstown State,Winthrop,NaN,-0.839898
74972,2025,Youngstown State,Wisconsin,NaN,-0.520170
74973,2025,Youngstown State,Wofford,NaN,-0.290202
74974,2025,Youngstown State,Wright State,0.659288,0.439871


In [44]:
df_match, team_to_spelling = match_names(df_spellings['TeamNameSpelling'].unique(), df_h2h['Team A'].unique())

df_match.head(25)

  0%|          | 0/364 [00:00<?, ?it/s]

,New Data Team,Team Spelling,Match Score
0,Abilene Christian,abilene christian,100
1,Radford,radford,100
2,Quinnipiac,quinnipiac,100
3,Queens (NC),Queens (NC),100
4,Purdue Fort Wayne,purdue fort wayne,100
5,Purdue,purdue,100
6,Providence,providence,100
7,Princeton,princeton,100
8,Presbyterian,presbyterian,100
9,Rhode Island,rhode island,100


In [45]:
df_h2h.insert(df_h2h.columns.get_loc('Team A'), 'Team A ID', df_h2h['Team A'].map(team_to_spelling).map(spelling_to_id))

df_h2h.insert(df_h2h.columns.get_loc('Team B'), 'Team B ID', df_h2h['Team B'].map(team_to_spelling).map(spelling_to_id))

df_h2h

,Season,Team A ID,Team A,Team B ID,Team B,Head to Head,Common Opps
0,2025,1101,Abilene Christian,1103,Akron,NaN,0.109367
1,2025,1101,Abilene Christian,1104,Alabama,NaN,-1.428608
2,2025,1101,Abilene Christian,1105,Alabama A&M,NaN,1.143470
3,2025,1101,Abilene Christian,1106,Alabama State,NaN,0.796956
4,2025,1101,Abilene Christian,1108,Alcorn State,NaN,1.306308
...,...,...,...,...,...,...,...
74971,2025,1464,Youngstown State,1457,Winthrop,NaN,-0.839898
74972,2025,1464,Youngstown State,1458,Wisconsin,NaN,-0.520170
74973,2025,1464,Youngstown State,1459,Wofford,NaN,-0.290202
74974,2025,1464,Youngstown State,1460,Wright State,0.659288,0.439871


In [46]:
df_mod = pd.merge(
    df_mod,
    df_h2h[['Season', 'Team A ID', 'Team B ID', 'Head to Head', 'Common Opps']],
    how='left',
    on=['Season', 'Team A ID', 'Team B ID'],
)

df_mod

,Season,Team A ID,Team B ID,Team A Region,Team B Region,Team A Seed,Team B Seed,Round,Head to Head,Common Opps
0,2025,1101,1102,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025,1101,1103,NaN,W,NaN,13.0,NaN,NaN,0.109367
2,2025,1101,1104,NaN,W,NaN,2.0,NaN,NaN,-1.428608
3,2025,1101,1105,NaN,NaN,NaN,NaN,NaN,NaN,1.143470
4,2025,1101,1106,NaN,Y,NaN,16.0,NaN,NaN,0.796956
...,...,...,...,...,...,...,...,...,...,...
144015,2025,1480,1475,NaN,NaN,NaN,NaN,NaN,NaN,0.452610
144016,2025,1480,1476,NaN,NaN,NaN,NaN,NaN,NaN,NaN
144017,2025,1480,1477,NaN,NaN,NaN,NaN,NaN,NaN,0.000000
144018,2025,1480,1478,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Get team names

In [47]:
df_teams = pd.read_csv(r'..\data\unprocessed\kaggle\MTeams.csv')

df_teams

,TeamID,TeamName,FirstD1Season,LastD1Season
0,1101,Abilene Chr,2014,2025
1,1102,Air Force,1985,2025
2,1103,Akron,1985,2025
3,1104,Alabama,1985,2025
4,1105,Alabama A&M,2000,2025
...,...,...,...,...
375,1476,Stonehill,2023,2025
376,1477,East Texas A&M,2023,2025
377,1478,Le Moyne,2024,2025
378,1479,Mercyhurst,2025,2025


In [48]:
id_to_team = dict(zip(df_teams['TeamID'], df_teams['TeamName']))

df_mod.insert(df_mod.columns.get_loc('Team A ID') + 1, 'Team A', df_mod['Team A ID'].map(id_to_team))
df_mod.insert(df_mod.columns.get_loc('Team B ID') + 1, 'Team B', df_mod['Team B ID'].map(id_to_team))

df_mod

,Season,Team A ID,Team A,Team B ID,Team B,Team A Region,Team B Region,Team A Seed,Team B Seed,Round,Head to Head,Common Opps
0,2025,1101,Abilene Chr,1102,Air Force,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2025,1101,Abilene Chr,1103,Akron,NaN,W,NaN,13.0,NaN,NaN,0.109367
2,2025,1101,Abilene Chr,1104,Alabama,NaN,W,NaN,2.0,NaN,NaN,-1.428608
3,2025,1101,Abilene Chr,1105,Alabama A&M,NaN,NaN,NaN,NaN,NaN,NaN,1.143470
4,2025,1101,Abilene Chr,1106,Alabama St,NaN,Y,NaN,16.0,NaN,NaN,0.796956
...,...,...,...,...,...,...,...,...,...,...,...,...
144015,2025,1480,West Georgia,1475,Southern Indiana,NaN,NaN,NaN,NaN,NaN,NaN,0.452610
144016,2025,1480,West Georgia,1476,Stonehill,NaN,NaN,NaN,NaN,NaN,NaN,NaN
144017,2025,1480,West Georgia,1477,East Texas A&M,NaN,NaN,NaN,NaN,NaN,NaN,0.000000
144018,2025,1480,West Georgia,1478,Le Moyne,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Map features

In [49]:
team_a_features = pd.merge(
    df_mod[['Season', 'Team A ID']],
    df.drop(columns=['Team', 'Seed', 'Region', 'Play In']),
    how='left',
    left_on=['Season', 'Team A ID'],
    right_on=['Season', 'TeamID'],
).drop(columns=['Season', 'Team A ID', 'TeamID'])

team_b_features = pd.merge(
    df_mod[['Season', 'Team B ID']],
    df.drop(columns=['Team', 'Seed', 'Region', 'Play In']),
    how='left',
    left_on=['Season', 'Team B ID'],
    right_on=['Season', 'TeamID'],
).drop(columns=['Season', 'Team B ID', 'TeamID'])

assert (df_mod.shape[0] == team_a_features.shape[0]), 'There is an issue with merging'
assert (df_mod.shape[0] == team_b_features.shape[0]), 'There is an issue with merging'

df_features = team_a_features - team_b_features

df_features['Team A ADJ OE Team B ADJ DE'] = team_a_features['ADJ OE'] + team_b_features['ADJ DE']
df_features['Team B ADJ OE Team A ADJ DE'] = team_b_features['ADJ OE'] + team_a_features['ADJ DE']

df_features['Team A Offense Team B Defense'] = team_a_features['Adjusted Offense'] + team_b_features['Adjusted Defense']
df_features['Team B Offense Team A Defense'] = team_b_features['Adjusted Offense'] + team_a_features['Adjusted Defense']

df_features['Team A BARTHAG'] = team_a_features['BARTHAG']
df_features['Team B BARTHAG'] = team_b_features['BARTHAG']

df_features

,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters,Mu,OS Rating,Team A ADJ OE Team B ADJ DE,Team B ADJ OE Team A ADJ DE,Team A Offense Team B Defense,Team B Offense Team A Defense,Team A BARTHAG,Team B BARTHAG
0,0.0,0.5,-0.6,-7.9,7.3,0.168,0.323276,-3.4,-3.2,1.8,15.9,-0.1,8.3,7.3,2.5,5.4,-2.2,-3.7,0.3,-1.2,-11.3,3.0,-23.2,-3.7,4.8,-0.917,0.182,16.513,8.4,-5.782,0.049,2.203,0.19200,7.47900,0.945660,0.083413,-0.000393,-0.083807,4.570764,0.304363,11.731447,13.564996,207.8,200.5,2.088581,2.005168,0.401,0.233
1,-1.0,0.0,-15.1,-4.6,-10.5,-0.275,-0.364224,-8.7,1.9,10.4,17.6,4.1,6.8,-3.3,2.5,-2.3,2.0,0.7,-1.4,4.2,-6.9,6.9,-20.1,-5.5,-2.0,2.127,-0.568,0.485,-3.0,4.743,-0.188,-6.822,-0.06950,-2.64550,-1.382271,-0.127239,-0.149890,-0.022651,-2.584935,-0.352560,-16.659737,-16.248421,204.5,215.0,2.027426,2.154665,0.401,0.676
2,-5.0,-2.5,-28.8,5.5,-34.3,-0.560,-0.309300,-9.6,3.5,-2.9,17.4,4.0,10.5,-4.7,2.3,-6.2,3.8,1.8,-1.7,-1.0,-2.5,7.8,-20.9,-2.6,-5.5,-2.946,-0.004,-41.859,0.3,-31.684,-0.515,-27.199,-0.37175,-20.92500,-3.048693,-0.349414,-0.271386,0.078028,-5.814190,-0.465910,-27.890797,-28.521589,194.4,228.7,1.926747,2.276161,0.401,0.961
3,0.0,0.5,5.4,-13.0,18.4,0.321,0.206897,1.6,-2.9,-1.4,4.6,-1.2,3.0,-3.9,-3.1,-1.7,0.8,-6.4,-2.3,2.5,-1.8,-6.2,-16.3,-5.0,-1.6,-0.730,-0.097,16.513,5.2,6.492,0.240,10.888,0.39050,16.36675,1.597966,0.181032,0.060491,-0.120541,-1.374740,0.304057,15.950441,15.861916,212.9,194.5,2.125316,1.944284,0.401,0.080
4,0.0,0.5,-3.1,-9.0,5.9,0.136,-0.110548,-0.4,0.8,8.1,12.8,7.6,5.3,1.7,0.8,1.1,2.1,-1.2,0.1,1.9,6.4,0.9,-17.5,-7.1,1.6,-0.043,-0.531,12.616,2.4,3.509,0.212,8.949,0.42025,19.65625,0.260927,0.023779,-0.028434,-0.052214,1.027856,-0.066933,0.003189,-0.237763,208.9,203.0,2.056988,2.033209,0.401,0.265
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144015,0.0,0.0,-1.5,0.2,-1.7,-0.024,-0.119048,-0.7,4.3,-1.9,0.7,-0.9,-0.1,-1.8,-2.1,0.8,6.9,0.1,2.2,-0.7,7.6,10.7,-3.4,7.2,1.1,-0.767,0.248,0.000,-4.9,5.132,NaN,NaN,NaN,NaN,-0.537673,-0.026032,-0.015381,0.010652,0.761904,-0.042736,-2.113372,-2.825316,211.4,213.1,2.095770,2.121802,0.131,0.155
144016,0.0,0.0,-3.1,0.3,-3.4,-0.051,-0.266667,-3.9,5.2,-2.6,3.6,-2.0,0.4,-0.3,-0.2,3.2,6.8,1.9,2.4,-5.1,-9.2,16.4,-14.4,10.4,2.9,-0.843,0.529,0.000,-3.1,3.459,NaN,NaN,NaN,NaN,-0.824001,-0.039006,-0.037384,0.001622,2.934996,-0.070972,-7.169788,-8.221590,211.3,214.7,2.104800,2.143806,0.131,0.182
144017,0.0,0.0,1.2,3.1,-1.9,-0.021,0.005376,-2.0,2.3,0.8,-0.2,-6.2,-3.3,-2.0,-6.7,1.1,4.3,-0.6,-1.4,-5.9,-10.4,-2.1,-18.2,9.1,0.6,-0.694,0.141,-8.001,4.6,-3.254,NaN,NaN,NaN,NaN,-0.443553,-0.022554,0.016073,0.038627,0.520975,0.060624,-1.384485,-1.175457,208.5,210.4,2.067794,2.090349,0.131,0.152
144018,0.0,0.0,-4.9,-6.3,1.4,0.006,-0.066667,-4.3,1.4,-8.9,-0.6,-3.3,0.3,0.4,-3.7,0.5,2.8,-0.2,3.4,-1.5,-3.1,0.1,-11.2,2.3,-0.5,-0.965,0.227,-8.578,-2.1,2.067,NaN,NaN,NaN,NaN,-0.271725,0.020275,-0.050100,-0.070374,0.110658,0.111673,0.182297,-0.758266,217.9,216.5,2.176796,2.156521,0.131,0.125


In [50]:
df_mod[df_features.columns] = df_features

df_mod

,Season,Team A ID,Team A,Team B ID,Team B,Team A Region,Team B Region,Team A Seed,Team B Seed,Round,Head to Head,Common Opps,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters,Mu,OS Rating,Team A ADJ OE Team B ADJ DE,Team B ADJ OE Team A ADJ DE,Team A Offense Team B Defense,Team B Offense Team A Defense,Team A BARTHAG,Team B BARTHAG
0,2025,1101,Abilene Chr,1102,Air Force,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.5,-0.6,-7.9,7.3,0.168,0.323276,-3.4,-3.2,1.8,15.9,-0.1,8.3,7.3,2.5,5.4,-2.2,-3.7,0.3,-1.2,-11.3,3.0,-23.2,-3.7,4.8,-0.917,0.182,16.513,8.4,-5.782,0.049,2.203,0.19200,7.47900,0.945660,0.083413,-0.000393,-0.083807,4.570764,0.304363,11.731447,13.564996,207.8,200.5,2.088581,2.005168,0.401,0.233
1,2025,1101,Abilene Chr,1103,Akron,NaN,W,NaN,13.0,NaN,NaN,0.109367,-1.0,0.0,-15.1,-4.6,-10.5,-0.275,-0.364224,-8.7,1.9,10.4,17.6,4.1,6.8,-3.3,2.5,-2.3,2.0,0.7,-1.4,4.2,-6.9,6.9,-20.1,-5.5,-2.0,2.127,-0.568,0.485,-3.0,4.743,-0.188,-6.822,-0.06950,-2.64550,-1.382271,-0.127239,-0.149890,-0.022651,-2.584935,-0.352560,-16.659737,-16.248421,204.5,215.0,2.027426,2.154665,0.401,0.676
2,2025,1101,Abilene Chr,1104,Alabama,NaN,W,NaN,2.0,NaN,NaN,-1.428608,-5.0,-2.5,-28.8,5.5,-34.3,-0.560,-0.309300,-9.6,3.5,-2.9,17.4,4.0,10.5,-4.7,2.3,-6.2,3.8,1.8,-1.7,-1.0,-2.5,7.8,-20.9,-2.6,-5.5,-2.946,-0.004,-41.859,0.3,-31.684,-0.515,-27.199,-0.37175,-20.92500,-3.048693,-0.349414,-0.271386,0.078028,-5.814190,-0.465910,-27.890797,-28.521589,194.4,228.7,1.926747,2.276161,0.401,0.961
3,2025,1101,Abilene Chr,1105,Alabama A&M,NaN,NaN,NaN,NaN,NaN,NaN,1.143470,0.0,0.5,5.4,-13.0,18.4,0.321,0.206897,1.6,-2.9,-1.4,4.6,-1.2,3.0,-3.9,-3.1,-1.7,0.8,-6.4,-2.3,2.5,-1.8,-6.2,-16.3,-5.0,-1.6,-0.730,-0.097,16.513,5.2,6.492,0.240,10.888,0.39050,16.36675,1.597966,0.181032,0.060491,-0.120541,-1.374740,0.304057,15.950441,15.861916,212.9,194.5,2.125316,1.944284,0.401,0.080
4,2025,1101,Abilene Chr,1106,Alabama St,NaN,Y,NaN,16.0,NaN,NaN,0.796956,0.0,0.5,-3.1,-9.0,5.9,0.136,-0.110548,-0.4,0.8,8.1,12.8,7.6,5.3,1.7,0.8,1.1,2.1,-1.2,0.1,1.9,6.4,0.9,-17.5,-7.1,1.6,-0.043,-0.531,12.616,2.4,3.509,0.212,8.949,0.42025,19.65625,0.260927,0.023779,-0.028434,-0.052214,1.027856,-0.066933,0.003189,-0.237763,208.9,203.0,2.056988,2.033209,0.401,0.265
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144015,2025,1480,West Georgia,1475,Southern Indiana,NaN,NaN,NaN,NaN,NaN,NaN,0.452610,0.0,0.0,-1.5,0.2,-1.7,-0.024,-0.119048,-0.7,4.3,-1.9,0.7,-0.9,-0.1,-1.8,-2.1,0.8,6.9,0.1,2.2,-0.7,7.6,10.7,-3.4,7.2,1.1,-0.767,0.248,0.000,-4.9,5.132,NaN,NaN,NaN,NaN,-0.537673,-0.026032,-0.015381,0.010652,0.761904,-0.042736,-2.113372,-2.825316,211.4,213.1,2.095770,2.121802,0.131,0.155
144016,2025,1480,West Georgia,1476,Stonehill,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,-3.1,0.3,-3.4,-0.051,-0.266667,-3.9,5.2,-2.6,3.6,-2.0,0.4,-0.3,-0.2,3.2,6.8,1.9,2.4,-5.1,-9.2,16.4,-14.4,10.4,2.9,-0.843,0.529,0.000,-3.1,3.459,NaN,NaN,NaN,NaN,-0.824001,-0.039006,-0.037384,0.001622,2.934996,-0.070972,-7.169788,-8.221590,211.3,214.7,2.104800,2.143806,0.131,0.182
144017,2025,1480,West Georgia,1477,East Texas A&M,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,0.0,0.0,1.2,3.1,-1.9,-0.021,0.005376,-2.0,2.3,0.8,-0.2,-6.2,-3.3,-2.0,-6.7,1.1,4.3,-0.6,-1.4,-5.9,-10.4,-2.1,-18.2,9.1,0.6,-0.694,0.141,-8.001,4.6,-3.254,NaN,NaN,NaN,NaN,-0.443553,-0.022554,0.016073,0.038627,0.520975,0.060624,-1.384485,-1.175457,208.5,210.4,2.067794,2.090349,0.131,0.152
144018,2025,1480,West Georgia,1478,Le Moyne,NaN,NaN,NaN,NaN,NaN,NaN,NaN,

In [51]:
# track if game is a tournament matchup for later
tournament_matchup = (df_mod['Team A Region'].notna()) & (df_mod['Team B Region'].notna())

tournament_matchup.sum()

4556

In [52]:
df_mod.insert(1, 'Round', df_mod.pop('Round'))

df_mod.drop(columns=['Team A Region', 'Team B Region', 'Team A Seed', 'Team B Seed'], inplace=True)

df_mod

,Season,Round,Team A ID,Team A,Team B ID,Team B,Head to Head,Common Opps,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,WIN%,EFG,EFG D.,FT RATE,FT RATE D,TOV%,TOV% D,O REB%,OP OREB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,3P RATE D,ADJ. T,EFF. HGT.,EXP.,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters,Mu,OS Rating,Team A ADJ OE Team B ADJ DE,Team B ADJ OE Team A ADJ DE,Team A Offense Team B Defense,Team B Offense Team A Defense,Team A BARTHAG,Team B BARTHAG
0,2025,NaN,1101,Abilene Chr,1102,Air Force,NaN,NaN,0.0,0.5,-0.6,-7.9,7.3,0.168,0.323276,-3.4,-3.2,1.8,15.9,-0.1,8.3,7.3,2.5,5.4,-2.2,-3.7,0.3,-1.2,-11.3,3.0,-23.2,-3.7,4.8,-0.917,0.182,16.513,8.4,-5.782,0.049,2.203,0.19200,7.47900,0.945660,0.083413,-0.000393,-0.083807,4.570764,0.304363,11.731447,13.564996,207.8,200.5,2.088581,2.005168,0.401,0.233
1,2025,NaN,1101,Abilene Chr,1103,Akron,NaN,0.109367,-1.0,0.0,-15.1,-4.6,-10.5,-0.275,-0.364224,-8.7,1.9,10.4,17.6,4.1,6.8,-3.3,2.5,-2.3,2.0,0.7,-1.4,4.2,-6.9,6.9,-20.1,-5.5,-2.0,2.127,-0.568,0.485,-3.0,4.743,-0.188,-6.822,-0.06950,-2.64550,-1.382271,-0.127239,-0.149890,-0.022651,-2.584935,-0.352560,-16.659737,-16.248421,204.5,215.0,2.027426,2.154665,0.401,0.676
2,2025,NaN,1101,Abilene Chr,1104,Alabama,NaN,-1.428608,-5.0,-2.5,-28.8,5.5,-34.3,-0.560,-0.309300,-9.6,3.5,-2.9,17.4,4.0,10.5,-4.7,2.3,-6.2,3.8,1.8,-1.7,-1.0,-2.5,7.8,-20.9,-2.6,-5.5,-2.946,-0.004,-41.859,0.3,-31.684,-0.515,-27.199,-0.37175,-20.92500,-3.048693,-0.349414,-0.271386,0.078028,-5.814190,-0.465910,-27.890797,-28.521589,194.4,228.7,1.926747,2.276161,0.401,0.961
3,2025,NaN,1101,Abilene Chr,1105,Alabama A&M,NaN,1.143470,0.0,0.5,5.4,-13.0,18.4,0.321,0.206897,1.6,-2.9,-1.4,4.6,-1.2,3.0,-3.9,-3.1,-1.7,0.8,-6.4,-2.3,2.5,-1.8,-6.2,-16.3,-5.0,-1.6,-0.730,-0.097,16.513,5.2,6.492,0.240,10.888,0.39050,16.36675,1.597966,0.181032,0.060491,-0.120541,-1.374740,0.304057,15.950441,15.861916,212.9,194.5,2.125316,1.944284,0.401,0.080
4,2025,NaN,1101,Abilene Chr,1106,Alabama St,NaN,0.796956,0.0,0.5,-3.1,-9.0,5.9,0.136,-0.110548,-0.4,0.8,8.1,12.8,7.6,5.3,1.7,0.8,1.1,2.1,-1.2,0.1,1.9,6.4,0.9,-17.5,-7.1,1.6,-0.043,-0.531,12.616,2.4,3.509,0.212,8.949,0.42025,19.65625,0.260927,0.023779,-0.028434,-0.052214,1.027856,-0.066933,0.003189,-0.237763,208.9,203.0,2.056988,2.033209,0.401,0.265
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144015,2025,NaN,1480,West Georgia,1475,Southern Indiana,NaN,0.452610,0.0,0.0,-1.5,0.2,-1.7,-0.024,-0.119048,-0.7,4.3,-1.9,0.7,-0.9,-0.1,-1.8,-2.1,0.8,6.9,0.1,2.2,-0.7,7.6,10.7,-3.4,7.2,1.1,-0.767,0.248,0.000,-4.9,5.132,NaN,NaN,NaN,NaN,-0.537673,-0.026032,-0.015381,0.010652,0.761904,-0.042736,-2.113372,-2.825316,211.4,213.1,2.095770,2.121802,0.131,0.155
144016,2025,NaN,1480,West Georgia,1476,Stonehill,NaN,NaN,0.0,0.0,-3.1,0.3,-3.4,-0.051,-0.266667,-3.9,5.2,-2.6,3.6,-2.0,0.4,-0.3,-0.2,3.2,6.8,1.9,2.4,-5.1,-9.2,16.4,-14.4,10.4,2.9,-0.843,0.529,0.000,-3.1,3.459,NaN,NaN,NaN,NaN,-0.824001,-0.039006,-0.037384,0.001622,2.934996,-0.070972,-7.169788,-8.221590,211.3,214.7,2.104800,2.143806,0.131,0.182
144017,2025,NaN,1480,West Georgia,1477,East Texas A&M,NaN,0.000000,0.0,0.0,1.2,3.1,-1.9,-0.021,0.005376,-2.0,2.3,0.8,-0.2,-6.2,-3.3,-2.0,-6.7,1.1,4.3,-0.6,-1.4,-5.9,-10.4,-2.1,-18.2,9.1,0.6,-0.694,0.141,-8.001,4.6,-3.254,NaN,NaN,NaN,NaN,-0.443553,-0.022554,0.016073,0.038627,0.520975,0.060624,-1.384485,-1.175457,208.5,210.4,2.067794,2.090349,0.131,0.152
144018,2025,NaN,1480,West Georgia,1478,Le Moyne,NaN,NaN,0.0,0.0,-4.9,-6.3,1.4,0.006,-0.066667,-4.3,1.4,-8.9,-0.6,-3.3,0.3,0.4,-3.7,0.5,2.8,-0.2,3.4,-1.5,-3.1,0.1,-11.2,2.3,-0.5,-0.965,0.227,-8.578,-2.1,2.067,NaN,NaN,NaN,NaN,-0.271725,0.020275,-0.050100,-0.070374,0

Drop features that were not used in the model

In [53]:
df_mod.drop(
    columns=[
        # 'Team A ADJ OE Team B ADJ DE',
        # 'Team B ADJ OE Team A ADJ DE',
        '3P RATE D',
        # 'Past Year ADJ EM',
        'EFF. HGT.',
        'EFG',
        'WIN%',
        # 'Efficiency Margin',
        'EXP.',
        'FT RATE D',
        'Team A Offense Team B Defense',
        'Team B Offense Team A Defense',
        # 'Past Year Tournament Result',
        'Round',
        'ADJ. T',
        'Head to Head',
        'EFG D.',

        # 'Team A BARTHAG',
        # 'Team B BARTHAG',

        # 'Past Year Tournament Result',

        # 'TOV%',
        # 'TOV% D',

        'OP OREB%',

        # 'Mu',
    ],
    inplace=True,
)

df_mod

,Season,Team A ID,Team A,Team B ID,Team B,Common Opps,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,FT RATE,TOV%,TOV% D,O REB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters,Mu,OS Rating,Team A ADJ OE Team B ADJ DE,Team B ADJ OE Team A ADJ DE,Team A BARTHAG,Team B BARTHAG
0,2025,1101,Abilene Chr,1102,Air Force,NaN,0.0,0.5,-0.6,-7.9,7.3,0.168,1.8,-0.1,8.3,7.3,5.4,-2.2,-3.7,0.3,-1.2,-11.3,3.0,-23.2,16.513,8.4,-5.782,0.049,2.203,0.19200,7.47900,0.945660,0.083413,-0.000393,-0.083807,4.570764,0.304363,11.731447,13.564996,207.8,200.5,0.401,0.233
1,2025,1101,Abilene Chr,1103,Akron,0.109367,-1.0,0.0,-15.1,-4.6,-10.5,-0.275,10.4,4.1,6.8,-3.3,-2.3,2.0,0.7,-1.4,4.2,-6.9,6.9,-20.1,0.485,-3.0,4.743,-0.188,-6.822,-0.06950,-2.64550,-1.382271,-0.127239,-0.149890,-0.022651,-2.584935,-0.352560,-16.659737,-16.248421,204.5,215.0,0.401,0.676
2,2025,1101,Abilene Chr,1104,Alabama,-1.428608,-5.0,-2.5,-28.8,5.5,-34.3,-0.560,-2.9,4.0,10.5,-4.7,-6.2,3.8,1.8,-1.7,-1.0,-2.5,7.8,-20.9,-41.859,0.3,-31.684,-0.515,-27.199,-0.37175,-20.92500,-3.048693,-0.349414,-0.271386,0.078028,-5.814190,-0.465910,-27.890797,-28.521589,194.4,228.7,0.401,0.961
3,2025,1101,Abilene Chr,1105,Alabama A&M,1.143470,0.0,0.5,5.4,-13.0,18.4,0.321,-1.4,-1.2,3.0,-3.9,-1.7,0.8,-6.4,-2.3,2.5,-1.8,-6.2,-16.3,16.513,5.2,6.492,0.240,10.888,0.39050,16.36675,1.597966,0.181032,0.060491,-0.120541,-1.374740,0.304057,15.950441,15.861916,212.9,194.5,0.401,0.080
4,2025,1101,Abilene Chr,1106,Alabama St,0.796956,0.0,0.5,-3.1,-9.0,5.9,0.136,8.1,7.6,5.3,1.7,1.1,2.1,-1.2,0.1,1.9,6.4,0.9,-17.5,12.616,2.4,3.509,0.212,8.949,0.42025,19.65625,0.260927,0.023779,-0.028434,-0.052214,1.027856,-0.066933,0.003189,-0.237763,208.9,203.0,0.401,0.265
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
144015,2025,1480,West Georgia,1475,Southern Indiana,0.452610,0.0,0.0,-1.5,0.2,-1.7,-0.024,-1.9,-0.9,-0.1,-1.8,0.8,6.9,0.1,2.2,-0.7,7.6,10.7,-3.4,0.000,-4.9,5.132,NaN,NaN,NaN,NaN,-0.537673,-0.026032,-0.015381,0.010652,0.761904,-0.042736,-2.113372,-2.825316,211.4,213.1,0.131,0.155
144016,2025,1480,West Georgia,1476,Stonehill,NaN,0.0,0.0,-3.1,0.3,-3.4,-0.051,-2.6,-2.0,0.4,-0.3,3.2,6.8,1.9,2.4,-5.1,-9.2,16.4,-14.4,0.000,-3.1,3.459,NaN,NaN,NaN,NaN,-0.824001,-0.039006,-0.037384,0.001622,2.934996,-0.070972,-7.169788,-8.221590,211.3,214.7,0.131,0.182
144017,2025,1480,West Georgia,1477,East Texas A&M,0.000000,0.0,0.0,1.2,3.1,-1.9,-0.021,0.8,-6.2,-3.3,-2.0,1.1,4.3,-0.6,-1.4,-5.9,-10.4,-2.1,-18.2,-8.001,4.6,-3.254,NaN,NaN,NaN,NaN,-0.443553,-0.022554,0.016073,0.038627,0.520975,0.060624,-1.384485,-1.175457,208.5,210.4,0.131,0.152
144018,2025,1480,West Georgia,1478,Le Moyne,NaN,0.0,0.0,-4.9,-6.3,1.4,0.006,-8.9,-3.3,0.3,0.4,0.5,2.8,-0.2,3.4,-1.5,-3.1,0.1,-11.2,-8.578,-2.1,2.067,NaN,NaN,NaN,NaN,-0.271725,0.020275,-0.050100,-0.070374,0.110658,0.111673,0.182297,-0.758266,217.9,216.5,0.131,0.125


Check that data follows same format as the data that the model was trained on

In [54]:
df_mod_training = pd.read_parquet(data_path)

assert all(df_mod_training.drop(columns=['Result']).columns == df_mod.columns), 'Columns do not match'

'Columns Match'

'Columns Match'

### Get Model Predictions

In [55]:
import pickle

with open(model_path, 'rb') as f:
    mod = pickle.load(f)

mod

LGBMClassifier(early_stopping_round=25, feature_fraction=0.45132150352038525,
               lambda_l1=0.4490083837956961, lambda_l2=2.5940134368868186,
               learning_rate=0.07452223617474929, max_depth=19, metric='rmse',
               min_child_samples=38,
               monotone_constraints=[1, 1, 1, 1, -1, 1, 1, -1, -1, 0, 1, 0, -1,
                                     -1, 1, -1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1,
                                     1, 1, -1, ...],
               n_estimators=1000, num_leaves=212, random_state=22,
               verbosity=-1)

In [56]:
X = df_mod.drop(columns=['Season', 'Team A ID', 'Team A', 'Team B ID', 'Team B'])

df_mod['Prediction'] = mod.predict_proba(X)[:, 1]

df_mod['Prediction']

0         0.735935
1         0.367148
2         0.057140
3         0.884593
4         0.696016
            ...   
144015    0.381733
144016    0.516041
144017    0.623768
144018    0.512159
144019    0.516591
Name: Prediction, Length: 144020, dtype: float64

In [63]:
(
    df_mod[['Season', 'Team A ID', 'Team A', 'Team B ID', 'Team B', 'Prediction']]
    .to_parquet(f'../data/simulations/mens/matchup_predictions_{season}.parquet')
)

'Done'

'Done'

Turn predictions into matchup matrix

In [57]:
# filter down to just tournament games
df_mod = df_mod.loc[tournament_matchup, :].reset_index(drop=True)

# filter out play-in losers
df_mod = df_mod.loc[
    (~df_mod['Team A ID'].isin(playin_losers)) & (~df_mod['Team B ID'].isin(playin_losers)), 
    :
].reset_index(drop=True)

df_mod

,Season,Team A ID,Team A,Team B ID,Team B,Common Opps,Past Year Tournament Result,Past 4 Years Tournament Results,ADJ OE,ADJ DE,ADJ EM,BARTHAG,FT RATE,TOV%,TOV% D,O REB%,RAW T,2P % D.,3P % D.,BLK %,BLKED %,AST %,OP AST %,3P RATE,TALENT,FT%,ELITE SOS,Past Year BARTHAG,Past Year ADJ EM,Past 4 Years BARTHAG,Past 4 Years ADJ EM,Rating,Efficiency Margin,Adjusted Offense,Adjusted Defense,Adjusted Tempo,Starters,Mu,OS Rating,Team A ADJ OE Team B ADJ DE,Team B ADJ OE Team A ADJ DE,Team A BARTHAG,Team B BARTHAG,Prediction
0,2025,1103,Akron,1104,Alabama,-0.783840,-4.0,-2.50,-13.7,10.1,-23.8,-0.285,-13.3,-0.1,3.7,-1.4,-3.9,1.8,1.1,-0.3,-5.2,4.4,0.9,-0.8,-42.344,3.3,-36.427,-0.327,-20.377,-0.302250,-18.279500,-1.666422,-0.222175,-0.121496,0.100679,-3.229255,-0.113350,-11.231060,-12.273169,209.5,233.3,0.676,0.961,0.218855
1,2025,1103,Akron,1106,Alabama St,0.338479,1.0,0.50,12.0,-4.4,16.4,0.411,-2.3,3.5,-1.5,5.0,3.4,0.1,-1.9,1.5,-2.3,13.3,-6.0,2.6,12.131,5.4,-1.234,0.400,15.771,0.489750,22.301750,1.643198,0.151018,0.121456,-0.029562,3.612791,0.285626,16.662926,16.010658,224.0,207.6,0.676,0.265,0.814330
2,2025,1103,Akron,1110,American Univ,NaN,1.0,0.50,11.5,-2.5,14.0,0.358,-2.2,-0.3,-0.5,9.5,8.7,-2.9,-1.9,4.4,-2.6,6.3,-5.9,0.1,16.028,-0.8,1.393,0.353,14.274,0.385500,15.573000,1.339313,0.126201,0.098758,-0.027443,8.275233,0.242993,10.255072,9.472405,222.1,208.1,0.676,0.318,0.747846
3,2025,1103,Akron,1112,Arizona,-0.143182,-2.0,-1.25,-10.0,10.0,-20.0,-0.269,-9.8,0.4,0.6,-2.6,2.2,3.4,-1.7,-2.2,0.3,1.8,-8.5,10.8,-50.040,-3.5,-33.947,-0.362,-24.039,-0.312500,-19.397750,-0.871730,-0.189493,-0.087680,0.101812,1.527040,-0.081996,-5.769691,-7.077124,209.6,229.6,0.676,0.945,0.130761
4,2025,1103,Akron,1116,Arkansas,NaN,1.0,-2.25,1.9,10.7,-8.8,-0.177,-8.7,0.1,-0.3,5.7,2.0,2.1,0.0,-5.6,-0.1,5.3,-3.9,8.7,-58.580,1.5,-26.386,-0.096,-3.939,-0.225750,-11.341750,-0.466953,-0.102562,0.007250,0.109812,2.137820,0.077652,-1.476120,-1.854412,208.9,217.7,0.676,0.853,0.378076
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4027,2025,1471,UC San Diego,1435,Vanderbilt,NaN,0.0,0.00,-5.7,-3.3,-2.4,-0.024,-0.2,-1.3,3.3,-6.6,-3.3,-5.3,-3.8,-3.9,-1.3,7.2,4.5,9.4,-22.699,-0.1,-24.393,0.134,5.057,-0.298750,-11.936500,-0.230638,0.036545,-0.024719,-0.061265,-3.124117,0.251474,9.712879,8.440592,213.3,215.7,0.835,0.859,0.400301
4028,2025,1471,UC San Diego,1458,Wisconsin,NaN,-1.0,-1.25,-9.9,0.7,-10.6,-0.101,0.0,-0.7,8.8,-2.4,-1.7,-0.6,0.1,1.4,-0.7,2.4,2.5,1.5,-10.670,-8.2,-23.899,-0.311,-16.705,-0.485500,-21.775000,-0.671902,-0.044252,-0.053779,-0.009528,-1.584322,0.018958,1.663777,-0.085035,209.3,219.9,0.835,0.936,0.168321
4029,2025,1471,UC San Diego,1459,Wofford,NaN,0.0,0.00,1.4,-12.1,13.5,0.301,2.7,-3.5,8.4,-9.9,2.3,-5.6,-1.4,-1.3,0.4,2.3,2.5,1.9,13.891,7.8,-1.000,0.199,7.622,-0.089000,-3.346500,1.010234,0.155847,0.020987,-0.134860,2.003840,0.274998,17.395726,16.091496,222.1,208.6,0.835,0.534,0.827383
4030,2025,1471,UC San Diego,1462,Xavier,NaN,0.0,-0.75,-3.6,-1.1,-2.5,-0.030,-3.1,-2.5,5.8,0.7,-2.5,-4.3,-1.1,0.4,-1.1,-6.2,-3.8,12.0,-28.829,-4.4,-17.497,-0.222,-9.689,-0.445000,-19.140750,0.230802,0.017972,-0.014272,-0.032245,-2.637067,0.169919,7.317106,6.194545,211.1,213.6,0.835,0.865,0.529455


In [58]:
df_matrix = (
    df_mod[['Team A ID', 'Team B ID', 'Prediction']]
    .pivot(
        index=['Team A ID'], 
        columns=['Team B ID'],
        values='Prediction',
    )
)

df_matrix

Team B ID,1103,1104,1106,1110,1112,1116,1120,1124,1136,1140,1155,1161,1163,1166,1179,1181,1188,1196,1208,1211,1213,1219,1222,1228,1235,1242,1246,1251,1252,1257,1266,1268,1270,1272,1276,1277,1279,1280,1281,1285,1303,1307,1313,1314,1328,1332,1345,1352,1385,1388,1397,1401,1403,1407,1417,1423,1429,1433,1435,1458,1459,1462,1463,1471
Team A ID,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1103,NaN,0.218855,0.814330,0.747846,0.130761,0.378076,0.108685,0.227940,0.400273,0.162867,0.141767,0.217477,0.182613,0.152973,0.433833,0.068494,0.708379,0.122202,0.206690,0.093120,0.385600,0.492275,0.069799,0.212648,0.198045,0.119670,0.161474,0.590858,0.661924,0.161206,0.118439,0.097387,0.300348,0.276252,0.171951,0.154397,0.243739,0.200797,0.127840,0.587032,0.656595,0.283579,0.623069,0.152199,0.271491,0.268023,0.184938,0.598475,0.120288,0.141557,0.075853,0.162088,0.106496,0.461942,0.225625,0.652260,0.384147,0.186827,0.286238,0.200268,0.754994,0.340162,0.511514,0.375963
1104,0.773840,NaN,0.895708,0.873932,0.554941,0.791982,0.365387,0.663334,0.920285,0.567786,0.567572,0.871512,0.819175,0.865018,0.887773,0.219921,0.921878,0.253007,0.666295,0.659649,0.915527,0.897161,0.305743,0.592925,0.761049,0.640653,0.739727,0.900529,0.896616,0.606808,0.509090,0.595208,0.899091,0.836064,0.558095,0.635498,0.761060,0.782011,0.778688,0.934768,0.935407,0.801104,0.949688,0.765542,0.891663,0.779390,0.583837,0.949091,0.518280,0.579855,0.471348,0.497814,0.448000,0.894360,0.716078,0.918126,0.926590,0.663348,0.886933,0.535712,0.883818,0.920449,0.881328,0.727097
1106,0.162046,0.090486,NaN,0.601517,0.043280,0.076725,0.074367,0.070406,0.187017,0.075485,0.117125,0.054602,0.099082,0.085230,0.126404,0.042574,0.423595,0.073230,0.051730,0.054667,0.110506,0.229729,0.045400,0.078741,0.073297,0.052883,0.068444,0.110275,0.148377,0.068074,0.041425,0.039706,0.078537,0.060967,0.074181,0.055188,0.083331,0.071666,0.058588,0.361113,0.331485,0.060918,0.271723,0.081803,0.081441,0.073306,0.056867,0.316398,0.056785,0.046170,0.066434,0.064029,0.067799,0.168806,0.103732,0.291040,0.137233,0.044921,0.062529,0.102626,0.404787,0.075956,0.128954,0.052860
1110,0.250457,0.120929,0.318411,NaN,0.050041,0.064909,0.047914,0.059204,0.411813,0.071242,0.057674,0.075373,0.061219,0.078504,0.103918,0.040944,0.464864,0.041666,0.080625,0.052089,0.084680,0.162147,0.033339,0.119439,0.067295,0.058435,0.082228,0.212605,0.255123,0.056002,0.083024,0.037071,0.069659,0.059988,0.093531,0.061328,0.080230,0.061903,0.065160,0.426689,0.352620,0.072665,0.345300,0.062826,0.072896,0.057413,0.063061,0.399764,0.044992,0.042264,0.068014,0.088811,0.068316,0.108859,0.082905,0.200782,0.184222,0.061313,0.076997,0.074019,0.430693,0.065576,0.125009,0.046498
1112,0.845163,0.411177,0.938787,0.939241,NaN,0.804638,0.235308,0.496646,0.868259,0.571583,0.372627,0.748614,0.664762,0.625196,0.855499,0.245570,0.931439,0.409921,0.570094,0.403422,0.861654,0.877436,0.217001,0.597953,0.679849,0.458624,0.521171,0.702235,0.885786,0.685157,0.430736,0.528988,0.847090,0.795267,0.696764,0.354440,0.646363,0.681545,0.453841,0.882985,0.904607,0.670231,0.918894,0.709664,0.751192,0.627955,0.454033,0.910176,0.550072,0.532861,0.290509,0.364034,0.418512,0.932662,0.519803,0.916217,0.717572,0.624988,0.662555,0.660604,0.893110,0.798711,0.813411,0.828993
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1458,0.757623,0.413574,0.889539,0.933357,0.389597,0.563831,0.176939,0.567463,0.836870,0.399985,0.452579,0.810910,0.556216,0.694846,0.805801,0.131407,0.926998,0.242592,0.339952,0.484925,0.782435,0.774019,0.096336,0.405367,0.563077,0.381417,0.530826,0.713493,0.843486,0.547589,0.332267,0.543858,0.785996,0.719824,0.468229,0.477838,0.648739,0.572130,0.490507,0.896755,0.910793,0.585822,0.904038,0.616691,0.578651,0.555757,0.417935,0.896250,0.461686,0.506062,0

In [59]:
df_matrix_display = df_matrix.copy()

df_matrix_display.columns = df_matrix_display.columns.map(id_to_team)
df_matrix_display.index = df_matrix_display.index.map(id_to_team)

df_matrix_display

Team B ID,Akron,Alabama,Alabama St,American Univ,Arizona,Arkansas,Auburn,Baylor,Bryant,BYU,Clemson,Colorado St,Connecticut,Creighton,Drake,Duke,SIUE,Florida,Georgia,Gonzaga,Grand Canyon,High Point,Houston,Illinois,Iowa St,Kansas,Kentucky,Liberty,Lipscomb,Louisville,Marquette,Maryland,McNeese St,Memphis,Michigan,Michigan St,Mississippi,Mississippi St,Missouri,Montana,NE Omaha,New Mexico,Norfolk St,North Carolina,Oklahoma,Oregon,Purdue,Robert Morris,St John's,St Mary's CA,Tennessee,Texas A&M,Texas Tech,Troy,UCLA,UNC Wilmington,Utah St,VCU,Vanderbilt,Wisconsin,Wofford,Xavier,Yale,UC San Diego
Team A ID,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
Akron,NaN,0.218855,0.814330,0.747846,0.130761,0.378076,0.108685,0.227940,0.400273,0.162867,0.141767,0.217477,0.182613,0.152973,0.433833,0.068494,0.708379,0.122202,0.206690,0.093120,0.385600,0.492275,0.069799,0.212648,0.198045,0.119670,0.161474,0.590858,0.661924,0.161206,0.118439,0.097387,0.300348,0.276252,0.171951,0.154397,0.243739,0.200797,0.127840,0.587032,0.656595,0.283579,0.623069,0.152199,0.271491,0.268023,0.184938,0.598475,0.120288,0.141557,0.075853,0.162088,0.106496,0.461942,0.225625,0.652260,0.384147,0.186827,0.286238,0.200268,0.754994,0.340162,0.511514,0.375963
Alabama,0.773840,NaN,0.895708,0.873932,0.554941,0.791982,0.365387,0.663334,0.920285,0.567786,0.567572,0.871512,0.819175,0.865018,0.887773,0.219921,0.921878,0.253007,0.666295,0.659649,0.915527,0.897161,0.305743,0.592925,0.761049,0.640653,0.739727,0.900529,0.896616,0.606808,0.509090,0.595208,0.899091,0.836064,0.558095,0.635498,0.761060,0.782011,0.778688,0.934768,0.935407,0.801104,0.949688,0.765542,0.891663,0.779390,0.583837,0.949091,0.518280,0.579855,0.471348,0.497814,0.448000,0.894360,0.716078,0.918126,0.926590,0.663348,0.886933,0.535712,0.883818,0.920449,0.881328,0.727097
Alabama St,0.162046,0.090486,NaN,0.601517,0.043280,0.076725,0.074367,0.070406,0.187017,0.075485,0.117125,0.054602,0.099082,0.085230,0.126404,0.042574,0.423595,0.073230,0.051730,0.054667,0.110506,0.229729,0.045400,0.078741,0.073297,0.052883,0.068444,0.110275,0.148377,0.068074,0.041425,0.039706,0.078537,0.060967,0.074181,0.055188,0.083331,0.071666,0.058588,0.361113,0.331485,0.060918,0.271723,0.081803,0.081441,0.073306,0.056867,0.316398,0.056785,0.046170,0.066434,0.064029,0.067799,0.168806,0.103732,0.291040,0.137233,0.044921,0.062529,0.102626,0.404787,0.075956,0.128954,0.052860
American Univ,0.250457,0.120929,0.318411,NaN,0.050041,0.064909,0.047914,0.059204,0.411813,0.071242,0.057674,0.075373,0.061219,0.078504,0.103918,0.040944,0.464864,0.041666,0.080625,0.052089,0.084680,0.162147,0.033339,0.119439,0.067295,0.058435,0.082228,0.212605,0.255123,0.056002,0.083024,0.037071,0.069659,0.059988,0.093531,0.061328,0.080230,0.061903,0.065160,0.426689,0.352620,0.072665,0.345300,0.062826,0.072896,0.057413,0.063061,0.399764,0.044992,0.042264,0.068014,0.088811,0.068316,0.108859,0.082905,0.200782,0.184222,0.061313,0.076997,0.074019,0.430693,0.065576,0.125009,0.046498
Arizona,0.845163,0.411177,0.938787,0.939241,NaN,0.804638,0.235308,0.496646,0.868259,0.571583,0.372627,0.748614,0.664762,0.625196,0.855499,0.245570,0.931439,0.409921,0.570094,0.403422,0.861654,0.877436,0.217001,0.597953,0.679849,0.458624,0.521171,0.702235,0.885786,0.685157,0.430736,0.528988,0.847090,0.795267,0.696764,0.354440,0.646363,0.681545,0.453841,0.882985,0.904607,0.670231,0.918894,0.709664,0.751192,0.627955,0.454033,0.910176,0.550072,0.532861,0.290509,0.364034,0.418512,0.932662,0.519803,0.916217,0.717572,0.624988,0.662555,0.660604,0.893110,0.798711,0.813411,0.828993
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Wisconsin,0.757623,0.413574,0.889539,0.933357,0.389597,0.563831,0.176939,0.567463,0.836870,0.399985,0.452579,0.810910,0.556216,0.694846,0.805801,0.131407,0.926998,0.24

In [60]:
df_matrix.to_csv(f'../data/simulations/mens/matchup_matrix_{season}.csv', index=True)
df_matrix_display.to_csv(f'../data/simulations/mens/matchup_matrix_display_{season}.csv', index=True)

'Done'

'Done'